# Conformal Prediction for Credit Risk under Temporal Data Shift
**Empirical benchmark notebook — real Lending Club data**

---

### Abstract
Credit-scoring models are trained and validated as if loans were exchangeable, yet portfolios drift and default labels arrive months to years after origination. We study what **split conformal prediction** guarantees when a LightGBM default model trained on 2007–2011 Lending Club loans is calibrated on 2012 and deployed out-of-time on 2013–2015, and which adaptation strategies restore validity when labels are delayed. We compare static split conformal, class-conditional (Mondrian) and risk-bin-conditional conformal, Adaptive Conformal Inference (ACI), DtACI, quantile tracking, sliding-window recalibration, retrain-and-recalibrate, recency-weighted conformal, and label-shift-weighted conformal that uses only unlabelled covariates. Prediction sets $\{0\}$ approve, $\{1\}$ reject and $\{0,1\}$ route to manual review; §20 compares them with cost-optimal cutoffs and trivial policies under an explicitly illustrative cost model.

**Data.** Real accepted-loan records from the public LendingClub extract (Kaggle, "All Lending Club loan data", CC0), restricted to 36-month loans with a final outcome (Fully Paid = 0; Charged Off/Default = 1) issued through 2015, so that every loan has had its full term to resolve before the 2018 Q4 extract (see `prepare_lending_club.py`). Results are generated by running the cells; §20d prints the key numbers and a draft results sentence from the live objects. **Write the abstract's findings from those outputs, not from memory.**

> **Status.** Run All on your machine. Cells with an empty output have no stored result yet; nothing in the text below is a result of your data until you have run it.


## Research questions and contributions
**RQ1 — Validity.** Does a split-conformal threshold calibrated on one year keep its nominal coverage out-of-time? (§4–5, §14, §16)
**RQ2 — Adaptation.** Which adaptive method restores validity when default labels are delayed: ACI, DtACI, quantile tracking, sliding-window recalibration, retrain-and-recalibrate, recency-weighted or label-shift-weighted conformal? (§6, §11–12b, §16b, §19, §15c)
**RQ3 — Diagnosis.** Is the failure driven by covariate shift, label (prior) shift or concept drift? (§15, §15b, §15c)
**RQ4 — Robustness.** Do the conclusions survive a different base learner, dropping the lender-assigned `int_rate`, within-month ordering, model-retraining noise and temporally dependent resampling? (§11b, §16b–c, §18c–e)
**RQ5 — Monitoring.** Can the loss of validity be detected early, with and without labels, at an acceptable false-alarm rate? (§20e)
**RQ6 — Utility.** Do conformal prediction sets improve underwriting decisions over cost-optimal cutoffs and trivial policies? (§8, §20–20b)

**Contributions (what this notebook adds beyond applying conformal prediction to a credit model).** (i) A delayed-label evaluation protocol in which only matured labels are used, with hindsight and sliding-window benchmarks (§12b). (ii) A label-free adaptation based on class-prior estimation, tested against oracle priors (§15c). (iii) Conditional-coverage diagnostics beyond the class (§13b).  (iv) A monitoring study with label-free (PSI, KS) and label-based (coverage test, calibration-in-the-large, conformal martingale) detectors, evaluated for false alarms on an exchangeable permutation null and for detection delay on the real stream (§20e). (v) Machine-generated tables and key numbers so that the text cannot drift from the results (§20d).

In [ ]:
import os, sys, subprocess, importlib
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Where things live: Desktop/Q2/archive holds the Kaggle files; results go to Desktop/Q2/outputs ----
from pathlib import Path
def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    cands = [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]
    for d in cands:
        if (d/'Q2').is_dir(): return d/'Q2'
    return None
Q2 = find_q2()
assert Q2 is not None, ("Could not find a folder called 'Q2' on your Desktop. Either move it there or set the path by hand, "
                        "e.g.  Q2 = Path(r'C:\\Users\\you\\Desktop\\Q2')  (replace this line).")
ARCHIVE = Q2 / 'archive'
print("Q2 folder:", Q2)

# Install only what is missing (works in externally-managed Pythons and on older pips).
def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        for extra in ([], ['--break-system-packages']):
            if subprocess.call([sys.executable, '-m', 'pip', '-q', 'install', pkg, *extra]) == 0:
                break
        importlib.invalidate_caches()
        importlib.import_module(module or pkg)

ensure('statsmodels'); ensure('lightgbm')

from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, precision_recall_curve, auc
import lightgbm as lgb

# Fix all sources of randomness and print package versions so results are traceable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

import sklearn
print(f"numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | lightgbm {lgb.__version__}")

# Aesthetics & Plotting Config
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Output directory is configurable via the OUTDIR environment variable.
OUTDIR = os.environ.get('OUTDIR', str(Q2 / 'outputs'))
os.makedirs(f'{OUTDIR}/figures', exist_ok=True)
os.makedirs(f'{OUTDIR}/tables', exist_ok=True)

def save_fig(name):
    """Every figure is saved as a 300-dpi PNG and a vector PDF (for the paper)."""
    plt.savefig(f'{OUTDIR}/figures/{name}.png', dpi=300, bbox_inches='tight'); plt.savefig(f'{OUTDIR}/figures/{name}.pdf', bbox_inches='tight')

print("Environment and packages initialized successfully.")

## 0. How to run this notebook
Just choose **Run All**. It expects the Kaggle files in `Desktop/Q2/archive` (set `Q2_DIR` to override). Step 0b reads `accepted_2007_to_2018Q4` (a `.csv`, a `.csv.gz`, or a `.zip` containing it), keeps **resolved 36-month loans issued up to 2015-12**, and caches the result as `Desktop/Q2/lending_club_real_36m.csv`, so later runs skip the slow read (a few minutes the first time). Tables and figures are written to `Desktop/Q2/outputs`.

Settings you may want to change are in the next cell and at the top of Section 0c: `MAX_PER_MONTH` (loans kept per issue month; 0 = all, slow) and `TERM` / `MAX_ISSUE`.


In [ ]:
# ---- 0b. Build the analysis file from the raw Kaggle extract (cached) ----
import gzip, hashlib, io, zipfile

TERM = 36                    # 36-month loans only: one population, and all of them resolve by the 2018-Q4 extract if issued <= 2015-12
MAX_ISSUE = '2015-12'        # latest issue month kept (36-month loans need 36 months to resolve)
FORCE_REBUILD = False        # True = ignore the cached CSV
PREPARED_CSV = Q2 / 'lending_club_real_36m.csv'

RAW_COLS = ['loan_amnt', 'term', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util',
            'delinq_2yrs', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_d', 'loan_status']
OUT_COLS = ['year', 'month', 'issue_d', 'default', 'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_score', 'revol_util',
            'delinq_2yrs', 'emp_length_yrs', 'home_ownership', 'verification_status', 'purpose']
GOOD, BAD = {'Fully Paid'}, {'Charged Off', 'Default'}

def find_raw():
    files = [p for p in ARCHIVE.rglob('*') if p.is_file()] if ARCHIVE.is_dir() else []
    acc = [p for p in files if 'accepted' in p.name.lower() and p.name.lower().endswith(('.csv', '.csv.gz'))]
    if acc: return max(acc, key=lambda p: p.stat().st_size)
    zips = [p for p in list(Q2.glob('*.zip')) + files if p.suffix.lower() == '.zip']
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            hit = [n for n in zf.namelist() if 'accepted' in n.lower() and n.lower().endswith(('.csv', '.csv.gz'))]
            if hit: return (z, hit[0])
    listing = "\n".join(str(p.relative_to(Q2)) for p in files[:30]) or "(folder missing or empty)"
    raise FileNotFoundError(f"No accepted-loans file (accepted_2007_to_2018Q4.csv / .csv.gz / .zip) under {ARCHIVE}. Found:\n{listing}")

def open_raw(raw):
    if isinstance(raw, tuple):
        zf = zipfile.ZipFile(raw[0]); f = zf.open(raw[1])
        return gzip.open(f) if raw[1].lower().endswith('.gz') else f
    return raw                                   # pandas reads .csv and .csv.gz paths directly

def build_prepared():
    raw = find_raw(); print("Reading raw file:", raw if not isinstance(raw, tuple) else f"{raw[0]} -> {raw[1]}")
    head = pd.read_csv(open_raw(raw), nrows=0)
    missing = [c for c in RAW_COLS if c not in head.columns]
    assert not missing, f"Raw file lacks columns {missing}. This notebook needs the 'accepted' file (it has FICO ranges)."
    parts, n_raw = [], 0
    for ch in pd.read_csv(open_raw(raw), usecols=RAW_COLS, chunksize=250_000, low_memory=False):
        n_raw += len(ch)
        ch = ch[ch['loan_status'].isin(GOOD | BAD)].copy()                      # final outcomes only
        ch['term'] = pd.to_numeric(ch['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
        parts.append(ch[ch['term'] == TERM])
    d = pd.concat(parts, ignore_index=True)
    print(f"Raw rows: {n_raw:,} | resolved {TERM}-month loans: {len(d):,}")
    d['issue_d'] = pd.to_datetime(d['issue_d'].astype(str).str.strip(), format='%b-%Y', errors='coerce')
    d = d.dropna(subset=['issue_d'])
    d = d[d['issue_d'] <= pd.Period(MAX_ISSUE).to_timestamp(how='end')]
    for c in ('int_rate', 'revol_util'):
        d[c] = pd.to_numeric(d[c].astype(str).str.rstrip('%'), errors='coerce')
    d['fico_score'] = (d['fico_range_low'] + d['fico_range_high']) / 2
    e = d['emp_length'].astype('string').str.strip()
    d['emp_length_yrs'] = e.str.extract(r'(\d+)')[0].astype('float')
    d.loc[e.str.startswith('<', na=False), 'emp_length_yrs'] = 0.0               # "< 1 year"
    d['default'] = d['loan_status'].isin(BAD).astype(int)
    d['year'], d['month'] = d['issue_d'].dt.year, d['issue_d'].dt.month
    out = d[OUT_COLS]; n0 = len(out); out = out.dropna()
    print(f"Dropped {n0 - len(out):,} rows with missing model inputs -> {len(out):,} rows")
    out = out.sort_values('issue_d', kind='stable').copy()
    out['issue_d'] = out['issue_d'].dt.strftime('%Y-%m-%d')
    out.to_csv(PREPARED_CSV, index=False)
    return out

if PREPARED_CSV.exists() and not FORCE_REBUILD:
    print("Using cached", PREPARED_CSV, "(set FORCE_REBUILD = True to rebuild from the raw file)")
else:
    build_prepared()
_chk = pd.read_csv(PREPARED_CSV, usecols=['year', 'default'])
print("\nLoans and default rate by issue year (full prepared file, before any per-month sampling):")
print(_chk.groupby('year')['default'].agg(N='size', default_rate='mean').round(4).T.to_string())
assert 0.05 < _chk['default'].mean() < 0.35, "Default rate looks wrong for real Lending Club 36-month loans - check the raw file."


In [ ]:
import hashlib

# Processed REAL data produced by prepare_lending_club.py (36-month loans, final outcomes, issued 2007-2015).
dataset_path = str(PREPARED_CSV)

# Max loans kept per issue month (even, seeded sampling). 0 / None = keep everything (slow for the online procedures).
MAX_PER_MONTH = int(os.environ.get('MAX_PER_MONTH', 2500))
LAST_TEST_YEAR = 2015          # loans issued later than this are not guaranteed to have matured labels

df = pd.read_csv(dataset_path)
REQUIRED_COLS = {'year', 'month', 'issue_d', 'default', 'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_score',
                 'revol_util', 'delinq_2yrs', 'emp_length_yrs', 'home_ownership', 'verification_status', 'purpose'}
assert REQUIRED_COLS <= set(df.columns), f"Missing columns: {REQUIRED_COLS - set(df.columns)}"
with open(dataset_path, 'rb') as fh:
    print(f"SHA-256 of {dataset_path}: {hashlib.sha256(fh.read()).hexdigest()}")
n_file = len(df)
df = df[df['year'] <= LAST_TEST_YEAR]
assert df['year'].max() == LAST_TEST_YEAR and df['year'].min() <= 2009, "unexpected year range for this design"

if MAX_PER_MONTH:
    df = df.sample(frac=1, random_state=SEED)                                   # seeded shuffle, then keep first K per month
    df = df[df.groupby(['year', 'month']).cumcount() < MAX_PER_MONTH]
print(f"Rows in file: {n_file:,} | after year filter and per-month cap ({MAX_PER_MONTH}): {len(df):,}")

print(f"Dataset Shape: {df.shape}")
print(f"Time Range: {df['year'].min()} to {df['year'].max()}")
print(f"Overall Default Rate: {df['default'].mean():.2%}")

# Sort by issue date (within-month order is the seeded random order) so every "online" procedure sees loans in issue order.
df['issue_d'] = pd.to_datetime(df['issue_d'])
df = df.sort_values('issue_d', kind='stable').reset_index(drop=True)
df['t_month'] = (df['year'] - 2007) * 12 + df['month']   # monthly time index
assert df['issue_d'].is_monotonic_increasing


## 1. Exploratory Data Analysis & Temporal Data Shift
We analyze key borrower attributes (Debt-to-Income ratio, FICO score, Interest Rate) across years to observe how the portfolio moves over time.

**Data provenance.** Real LendingClub accepted loans (Kaggle: *All Lending Club loan data*, N. George; public domain, 2007 to 2018 Q4). Selection (done in `prepare_lending_club.py`): 36-month term; final status only (Fully Paid, Charged Off, Default); issued on or before 2015-12 so that all loans are resolved (right-censoring removed); complete cases on the 11 model features. `fico_score` is the mean of the FICO range bounds. Loans are sampled evenly by issue month to at most `MAX_PER_MONTH` per month (set in the loading cell; `0` keeps everything) to keep the online procedures tractable. Because the lender's own `int_rate` is a function of its risk grade, see §2b and §18e.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='year', y='dti', ax=axes[0], hue='year', palette='Blues', legend=False)
axes[0].set_title('Debt-to-Income (DTI) Shift over Years', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='year', y='fico_score', ax=axes[1], hue='year', palette='Greens', legend=False)
axes[1].set_title('FICO Score Distribution over Years', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

sns.lineplot(data=df.groupby('year')['default'].mean().reset_index(), x='year', y='default', marker='o', color='crimson', linewidth=2.5, ax=axes[2])
axes[2].set_title('Default Rate Trend (2007-2015)', fontweight='bold')
axes[2].set_ylabel('Default Rate')

plt.tight_layout()
save_fig('01_eda_shift')
plt.show()


## 2. Temporal Data Splitting & Feature Encoding
To mimic a real-world enterprise deployment:
* **Train Set (2007–2011):** Historical model training
* **Calibration Set (2012):** Hold-out calibration for Conformal Thresholding
* **Test Set (2013–2015):** Out-of-Time (OOT) evaluation under distribution shift

The sample sizes of the three sets are printed by the next cell; quote those numbers in the paper.


In [ ]:
# Categorical Feature One-Hot Encoding
feature_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_score', 'revol_util',
                'delinq_2yrs', 'emp_length_yrs', 'home_ownership', 'verification_status', 'purpose']

df_encoded = pd.get_dummies(df[['year', 'default'] + feature_cols], columns=['home_ownership', 'verification_status', 'purpose'], drop_first=True)
X_cols = [c for c in df_encoded.columns if c not in ['year', 'default']]

# Define temporal splits
train_mask = df_encoded['year'] <= 2011
calib_mask = df_encoded['year'] == 2012
test_mask = df_encoded['year'] >= 2013

X_train, y_train = df_encoded.loc[train_mask, X_cols], df_encoded.loc[train_mask, 'default']
X_calib, y_calib = df_encoded.loc[calib_mask, X_cols], df_encoded.loc[calib_mask, 'default']
X_test, y_test = df_encoded.loc[test_mask, X_cols], df_encoded.loc[test_mask, 'default']

test_years = df_encoded.loc[test_mask, 'year'].values

print(f"Train samples (2007-2011): {X_train.shape[0]}")
print(f"Calibration samples (2012): {X_calib.shape[0]}")
print(f"Test samples (2013-2015): {X_test.shape[0]}")


## 2b. Data-integrity checks
Three assumptions underlie every downstream step and are tested here: `year`/`month` agree with `issue_d` (the monthly index drives every ACI delay), there are no missing values, and no duplicate loans leak across the temporal splits. A fourth — that labels are fully matured for the last years — cannot be checked on this file (a real Lending Club extract right-censors recent vintages).

In [ ]:
assert (df['issue_d'].dt.year == df['year']).all() and (df['issue_d'].dt.month == df['month']).all(), \
    "year/month disagree with issue_d: t_month (used for ACI label delays) would be wrong"
assert set(np.unique(df['default'])) <= {0, 1}
miss = df[feature_cols + ['default']].isna().sum()
print("Missing values in used columns:", miss[miss > 0].to_dict() or "none")

key = df[feature_cols].astype(str).agg('|'.join, axis=1).values
split = np.where(train_mask.values, 'train', np.where(calib_mask.values, 'calib', 'test'))
kd = pd.DataFrame({'key': key, 'split': split})
dups = kd[kd.duplicated('key', keep=False)]
print(f"Rows whose full feature vector is duplicated anywhere: {len(dups)}; keys spanning >1 split: "
      f"{int((dups.groupby('key')['split'].nunique() > 1).sum())}")
print("\nRows and default rate by year:")
print(df.groupby('year')['default'].agg(N='size', default_rate='mean').round(4).T.to_string())
print("\nNote: `int_rate` is a model input. In real lending data it is assigned by the lender's own risk model, so using it can make a"
      "\n'credit-risk model' partly a re-statement of the lender's earlier score; report results with and without it before publishing.")

## 3. LightGBM Base Model Training & Point Metric Evaluation
We train a LightGBM Gradient Boosted Decision Tree model on historical data (2007–2011) and evaluate point metrics (ROC-AUC, Log-Loss, Brier Score) across testing years.


In [ ]:
# Fit LightGBM Classifier
lgb_model = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)

# Predict probabilities on Train, Calibration, and Test
p_train = lgb_model.predict_proba(X_train)[:, 1]
p_calib = lgb_model.predict_proba(X_calib)[:, 1]
p_test = lgb_model.predict_proba(X_test)[:, 1]

print(f"Train ROC-AUC: {roc_auc_score(y_train, p_train):.4f}")
print(f"Calibration (2012) ROC-AUC: {roc_auc_score(y_calib, p_calib):.4f}")
print(f"OOT Test (2013-2015) ROC-AUC: {roc_auc_score(y_test, p_test):.4f}")

# Evaluate Performance Decay Year by Year
yearly_perf = []
for yr in range(2012, 2016):
    yr_mask = df_encoded['year'] == yr
    X_yr, y_yr = df_encoded.loc[yr_mask, X_cols], df_encoded.loc[yr_mask, 'default']
    p_yr = lgb_model.predict_proba(X_yr)[:, 1]

    auc_val = roc_auc_score(y_yr, p_yr)
    brier = brier_score_loss(y_yr, p_yr)
    logloss = log_loss(y_yr, p_yr)
    yearly_perf.append({'Year': yr, 'ROC-AUC': auc_val, 'Brier Score': brier, 'Log Loss': logloss})

df_perf = pd.DataFrame(yearly_perf)
print("Yearly Point Metric Performance Decay:")
print(df_perf.to_string(index=False))

# AUC need not fall monotonically (2013 beats the 2012 calibration year): ranking ability is roughly flat/noisy, while Brier score and
# log-loss decay steadily. That is a calibration story, not a discrimination story, so it is reported as such.
auc_series = df_perf.set_index('Year')['ROC-AUC']
brier_series = df_perf.set_index('Year')['Brier Score']
auc_monotonic = all(auc_series.iloc[i] >= auc_series.iloc[i+1] for i in range(len(auc_series)-1))
brier_monotonic = all(brier_series.iloc[i] <= brier_series.iloc[i+1] for i in range(len(brier_series)-1))
print(f"\nAUC monotonically decreasing across {list(auc_series.index)}? {auc_monotonic}")
print(f"Brier score monotonically increasing (worsening) across {list(brier_series.index)}? {brier_monotonic}")
print("-> Compare how AUC and the calibration metrics (Brier, log-loss) move across years.")

df_perf.round(4).to_csv(f'{OUTDIR}/tables/03_yearly_point_metrics.csv', index=False)


## 4. Split Conformal Prediction Implementation
Conformal Prediction produces **prediction sets** $C(X) \subseteq \{0, 1\}$ such that the true label $Y \in C(X)$ with probability at least $1 - \alpha$ — **provided calibration and test data are exchangeable**. Section 5 onward examines what happens when they are not.

We define the Non-Conformity Score $s(x, y)$ for classification:
$$s(x, y) = 1 - \hat{P}(Y = y \mid X = x)$$

The conformal threshold $q_{1-\alpha}$ is the $\lceil (N_{calib} + 1)(1 - \alpha)\rceil$-th **smallest** calibration score (an exact order statistic; equivalently the $\frac{\lceil (N_{calib}+1)(1-\alpha)\rceil}{N_{calib}}$ empirical quantile).

> `conf_q` below is the single implementation used by every section. It is **not** `np.quantile(..., method='higher')`: NumPy interpolates at index `q_level·(n−1)`, so that call returns the $(k{+}1)$-th order statistic (rank 2,488 instead of 2,487 for $N=2{,}762$), a threshold one rank too conservative.


In [ ]:
def conf_q(scores, alpha):
    """Finite-sample split-conformal quantile: the ceil((n+1)(1-alpha))-th smallest score
    (np.inf if that rank exceeds n). Single implementation used by every section."""
    s = np.sort(np.asarray(scores)); n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]

def compute_split_conformal_threshold(y_calib_true, p_calib_hat, alpha=0.10):
    # Non-conformity scores for true labels in calibration set: s_i = 1 - p(y_i | x_i)
    p_true = np.where(y_calib_true == 1, p_calib_hat, 1 - p_calib_hat)
    scores = 1.0 - p_true
    return conf_q(scores, alpha), scores

alpha_target = 0.10 # 90% Coverage Guarantee Target
q_hat, calib_scores = compute_split_conformal_threshold(y_calib.values, p_calib, alpha=alpha_target)

print(f"Target Coverage Level (1 - alpha): {1 - alpha_target:.2%}")
print(f"Computed Conformal Threshold q_hat: {q_hat:.4f}")

In [ ]:
def generate_conformal_prediction_sets(p_hat, q_hat):
    # Include class y in prediction set if (1 - p_hat(y)) <= q_hat => p_hat(y) >= 1 - q_hat
    threshold = 1.0 - q_hat

    sets = []
    for p in p_hat:
        s = set()
        if (1 - p) >= threshold: # Class 0
            s.add(0)
        if p >= threshold: # Class 1
            s.add(1)
        sets.append(s)
    return sets

# Evaluate on Calibration Set (2012)
calib_sets = generate_conformal_prediction_sets(p_calib, q_hat)
calib_coverage = np.mean([y_calib.values[i] in calib_sets[i] for i in range(len(y_calib))])
calib_set_sizes = [len(s) for s in calib_sets]

print(f"Empirical Coverage on Calibration Set (2012): {calib_coverage:.2%}")
print(f"Average Prediction Set Size (2012): {np.mean(calib_set_sizes):.3f}")


## 5. Testing Conformal Coverage Guarantees Under Temporal Data Shift
Now we evaluate whether the static conformal threshold $q_{1-\alpha}$ calculated on 2012 data retains its theoretical $90\%$ coverage guarantee when applied to out-of-time test years (2013, 2014, 2015).


In [ ]:
from statsmodels.stats.proportion import proportion_confint

yearly_conformal_res = []
covered_by_year = {}

for yr in range(2012, 2016):
    yr_mask = df_encoded['year'] == yr
    X_yr, y_yr = df_encoded.loc[yr_mask, X_cols], df_encoded.loc[yr_mask, 'default'].values
    p_yr = lgb_model.predict_proba(X_yr)[:, 1]

    sets_yr = generate_conformal_prediction_sets(p_yr, q_hat)
    covered = np.array([y_yr[i] in sets_yr[i] for i in range(len(y_yr))])
    covered_by_year[yr] = covered
    cov = covered.mean()
    n = len(covered)
    lo, hi = proportion_confint(covered.sum(), n, alpha=0.05, method='wilson')
    avg_size = np.mean([len(s) for s in sets_yr])

    set_counts = pd.Series([str(sorted(list(s))) for s in sets_yr]).value_counts(normalize=True).to_dict()

    yearly_conformal_res.append({
        'Year': yr,
        'N': n,
        'Empirical Coverage': cov,
        'CI Low (95%)': lo,
        'CI High (95%)': hi,
        'Target Coverage': 1.0 - alpha_target,
        'Coverage Gap': cov - (1.0 - alpha_target),
        'Avg Set Size': avg_size,
        'Empty Sets {}': set_counts.get('[]', 0.0),
        'Singletons {0}': set_counts.get('[0]', 0.0),
        'Singletons {1}': set_counts.get('[1]', 0.0),
        'Both {0, 1}': set_counts.get('[0, 1]', 0.0)
    })

df_cov = pd.DataFrame(yearly_conformal_res)

# Per-year 95% CIs: a year whose CI contains 0.90 is statistically consistent with nominal coverage even if its point estimate is a little below.
# The pooled figure covers the OUT-OF-TIME years 2013-2015 only: 2012 is the calibration year, so its coverage is ~nominal by construction.
oot_covered = np.concatenate([covered_by_year[yr] for yr in (2013, 2014, 2015)])
p_lo, p_hi = proportion_confint(oot_covered.sum(), len(oot_covered), alpha=0.05, method='wilson')

print(df_cov[['Year', 'N', 'Empirical Coverage', 'CI Low (95%)', 'CI High (95%)', 'Target Coverage', 'Coverage Gap', 'Avg Set Size']].round(4).to_string(index=False))
print(f"\nPooled 2013-2015 (out-of-time) coverage: {oot_covered.mean():.4f} (95% CI: {p_lo:.4f}-{p_hi:.4f})  [report only alongside the per-year rows above]")
for _, row in df_cov.iterrows():
    verdict = "consistent with 90% target" if row['CI Low (95%)'] <= 0.90 <= row['CI High (95%)'] else "significantly below 90% target"
    tag = "  <- calibration year, in-sample" if int(row['Year']) == 2012 else ""
    print(f"  {int(row['Year'])}: {row['Empirical Coverage']:.1%} (95% CI {row['CI Low (95%)']:.1%}-{row['CI High (95%)']:.1%}) -> {verdict}{tag}")

df_cov.round(4).to_csv(f'{OUTDIR}/tables/05_yearly_coverage_with_CI.csv', index=False)

## 6. Restoring Valid Coverage: Adaptive Conformal Inference (ACI)
Under severe data shift, static conformal coverage degrades. **Adaptive Conformal Inference (ACI)** (Gibbs & Candès, 2021) dynamically updates the target misclassification rate $\alpha_t$ online:
$$\alpha_{t+1} = \alpha_t + \gamma (\alpha - \text{err}_t)$$
where $\text{err}_t = 1$ if $y_t\notin C_{\alpha_t}(x_t)$ else $0$, and $\gamma > 0$ is the learning rate step-size.


In [ ]:
def run_adaptive_conformal_inference(X_test, y_test, model, y_calib, p_calib, target_alpha=0.10, gamma=0.02, clip=(0.01, 0.99)):
    """Online ACI with immediate feedback. Same exact-order-statistic quantile as conf_q; the alpha_t *state* is clipped every step,
    exactly like the vectorised `aci()` helper of Section 10 (Section 11 asserts the two are identical). alpha_history[t] is the alpha_t USED at step t."""
    p_test_all = model.predict_proba(X_test)[:, 1]

    current_alpha = target_alpha
    aci_coverages, alpha_history, set_sizes = [], [], []

    # Calibration set scores, sorted once (quantile lookup = k-th order statistic)
    p_calib_true = np.where(np.asarray(y_calib) == 1, p_calib, 1 - p_calib)
    s_sorted = np.sort(1.0 - p_calib_true)
    n_cal = len(s_sorted)

    for t in range(len(y_test)):
        alpha_history.append(current_alpha)
        k = int(np.ceil((n_cal + 1) * (1 - current_alpha)))
        q_t = s_sorted[min(k, n_cal) - 1]

        # Prediction set for test instance t: class 0 iff p <= q, class 1 iff 1-p <= q
        p_t, y_t = p_test_all[t], y_test[t]
        in0, in1 = p_t <= q_t, (1 - p_t) <= q_t
        covered = int(in1 if y_t == 1 else in0)
        err_t = 1 - covered

        # ACI update rule (state clipped every step)
        current_alpha = float(np.clip(current_alpha + gamma * (target_alpha - err_t), *clip))

        aci_coverages.append(covered)
        set_sizes.append(int(in0) + int(in1))

    return np.array(aci_coverages), np.array(alpha_history), np.array(set_sizes)

aci_covs, alpha_hist, aci_sizes = run_adaptive_conformal_inference(
    X_test, y_test.values, lgb_model, y_calib.values, p_calib, target_alpha=0.10, gamma=0.015
)

# Pooled coverage is sample-size weighted (not the unweighted mean of the yearly values).
oot = df_cov[df_cov['Year'] >= 2013]
static_pooled = np.average(oot['Empirical Coverage'], weights=oot['N'])
print(f"Static Conformal Coverage on Test Set (2013-2015, pooled): {static_pooled:.2%}")
print(f"Adaptive Conformal Inference (ACI) Overall Coverage: {np.mean(aci_covs):.2%}")
print(f"Target Coverage: {1 - alpha_target:.2%}")

## 7. Comparative Visualizations & Conformal Analysis
We plot static vs. adaptive conformal coverage over time alongside prediction set distributions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Empirical Coverage by Year
axes[0].plot(df_cov['Year'], df_cov['Empirical Coverage'], marker='o', linewidth=2.5, color='darkred', label='Static Conformal Coverage')
axes[0].axhline(0.90, linestyle='--', color='black', linewidth=2, label='Target Guarantee (90%)')
axes[0].set_ylim(0.80, 0.96); axes[0].set_xticks(list(df_cov['Year'])); axes[0].set_xlabel('Issue year')
axes[0].set_title('Conformal Coverage Degradation under Data Shift', fontweight='bold')
axes[0].set_ylabel('Coverage Rate')
axes[0].legend(loc='lower left')

# Plot 2: ACI Rolling Coverage (2,500-loan window ~ one month of loans)
rolling_aci = pd.Series(aci_covs).rolling(window=2500, min_periods=250).mean()
axes[1].plot(rolling_aci, color='teal', linewidth=1.5, label='Adaptive Conformal (ACI) Rolling Coverage')
axes[1].axhline(0.90, linestyle='--', color='black', linewidth=2, label='Target Guarantee (90%)')
axes[1].set_title('Adaptive Conformal Inference (ACI) Online Recovery', fontweight='bold')
axes[1].set_xlabel('Out-of-Time Test Instance (chronological, sorted by issue_d)')
axes[1].set_ylabel('Rolling Coverage Rate')
axes[1].set_ylim(0.86, 0.94); axes[1].legend(loc='lower right')

plt.tight_layout()
save_fig('07_coverage_and_aci')
plt.show()

## 8. Enterprise Credit Decisioning Framework
Conformal prediction sets directly translate to actionable underwriting strategies:
* **Set $\{0\}$ (Low Risk):** **Auto-Approve Loan** (High certainty of non-default).
* **Set $\{1\}$ (High Risk):** **Auto-Reject Loan** (High certainty of default).
* **Set $\{0, 1\}$ (Uncertain / Borderline):** **Route to Manual Underwriter Review** (Model requested human intervention).

> *Superseded by Section 20.* This first cut pools 2013–2015, uses the marginal (static) sets and has no cost model or baseline. Section 20 redoes it per year with Mondrian sets, a plain-cutoff baseline and trivial policies.

In [ ]:
test_sets = generate_conformal_prediction_sets(p_test, q_hat)
set_types = [str(sorted(list(s))) for s in test_sets]
df_test_res = pd.DataFrame({
    'fico_score': X_test['fico_score'].values,
    'dti': X_test['dti'].values,
    'p_default': p_test,
    'set_type': set_types,
    'actual_default': y_test.values
})

# Summary of Underwriting Actions
action_summary = df_test_res.groupby('set_type').agg(
    count=('p_default', 'count'),
    pct=('p_default', lambda x: len(x) / len(df_test_res)),
    actual_default_rate=('actual_default', 'mean'),
    avg_fico=('fico_score', 'mean'),
    avg_dti=('dti', 'mean')
).reset_index()

action_summary['Action Strategy'] = action_summary['set_type'].map({
    '[0]': 'Auto-Approve (Low Risk)',
    '[1]': 'Auto-Reject (High Risk)',
    '[0, 1]': 'Manual Underwriting (Uncertain)'
})

print("Enterprise Credit Decisioning Distribution (2013-2015 Test Horizon):")
print(action_summary[['Action Strategy', 'count', 'pct', 'actual_default_rate', 'avg_fico', 'avg_dti']].to_string(index=False))


## 9. Roadmap of the analyses
Sections 1–8 set up the data, the base model, split conformal, ACI and a first decision framework; the data are sorted by `issue_d` and every section uses the single exact-order-statistic quantile `conf_q`. The sections below test, extend and stress those results.

| § | Question | Analysis |
|---|----------|----------|
| 11, 11b | How does ACI behave on a chronological stream? | ACI on ordered data; $\gamma$ sensitivity; unclipped ACI vs. the Gibbs–Candès bound; within-month ordering robustness |
| 12, 12b | What if labels are delayed? | Delayed-feedback ACI; lagged recalibration; sliding-window recalibration, quantile tracking, hindsight benchmark, local coverage |
| 13, 13b | Who is under-covered? | Class-conditional (Mondrian) conformal; coverage by risk decile and borrower slices; bin-conditional conformal |
| 14 | Is the failure due to shift? | I.I.D. control experiment |
| 15, 15b, 15c | What kind of shift is it? | Calibration-in-the-large; domain classifiers; covariate-shift weights (in-sample and cross-fitted); label-free prior-shift estimation and weighting |
| 16, 16b, 16c | How certain are the coverage numbers? | Bootstrap; calibration-year / window sweep; combined calibration+test bootstrap; retrain-and-recalibrate; month-block bootstrap; walk-forward $\gamma$; model-retraining variance |
| 17 | Does the conclusion depend on $\alpha$? | $\alpha \in \{0.05, 0.10, 0.20\}$ sweep |
| 18, 18c–e | Does it depend on the base model or on `int_rate`? | Logistic-regression baseline; post-hoc calibration; conformal on logistic regression, on a validation-tuned LightGBM, and without `int_rate` |
| 19, 19c | Do DtACI / recency-weighted conformal help? | DtACI; Barber et al. weights; walk-forward selection of $\eta$, $\rho$ |
| 20, 20b | Do the sets improve underwriting decisions? | Per-year tiers; illustrative cost model; cost-optimal cutoffs and trivial policies; $\alpha$ tuned on 2012; margin sensitivity |
| 20e | Can the failure be detected early, and at what false-alarm rate? | PSI, KS, coverage test, calibration-in-the-large test, conformal martingale; permutation null |
| 20d | | LaTeX tables, key numbers and a draft results sentence generated from the objects above |
| 21, 22 | | Conclusions; references and pre-submission checklist |


## 10. Shared helpers
Vectorised helpers (`make_sets`, `evaluate`, `aci`) and the arrays reused by every later section; `conf_q` (Section 4) is the only quantile implementation.

In [ ]:
# ---------- shared helpers (vectorised) ----------
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
# Scope warning suppression narrowly (known-safe, noisy ones) so genuine issues (convergence, NaNs, deprecations) stay visible.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*use_label_encoder.*")

# conf_q(scores, alpha) is defined once, in Section 4.

def make_sets(p, q0, q1=None):
    """Boolean (n,2) membership. Score(x,0)=p, score(x,1)=1-p. q1 defaults to q0 (marginal)."""
    q1 = q0 if q1 is None else q1
    return np.column_stack([p <= q0, (1 - p) <= q1])

def evaluate(sets, y):
    y = np.asarray(y).astype(int)
    covered = sets[np.arange(len(y)), y]
    return dict(cov=covered.mean(), cov_nd=covered[y == 0].mean(), cov_d=covered[y == 1].mean(),
                size=sets.sum(1).mean())

def true_label_scores(p, y):
    return np.where(np.asarray(y) == 1, 1 - p, p)      # = 1 - P_hat(y|x)

def aci(p, y, month, cal_scores, alpha=0.10, gamma=0.015, delay=0, clip=(0.01, 0.99)):
    """ACI over a chronologically sorted stream. Label of loan j is revealed `delay`
    months after its issue month; alpha_t is only updated with revealed labels.
    The alpha_t *state* is clipped to `clip` every step (not only where it is converted to a quantile level), so the reported alpha_t
    history stays interpretable under delay and large gamma. Clipping voids the Gibbs-Candes bound; see Section 11b for the unclipped version."""
    s = np.sort(cal_scores); n = len(s); T = len(y)
    a = alpha; err = np.zeros(T); size = np.zeros(T); a_hist = np.zeros(T); ptr = 0
    for t in range(T):
        while ptr < t and month[ptr] + delay <= month[t]:
            a = np.clip(a + gamma * (alpha - err[ptr]), *clip); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a)))
        q = s[min(k, n) - 1]
        in0, in1 = p[t] <= q, (1 - p[t]) <= q
        err[t] = 1 - (in1 if y[t] == 1 else in0); size[t] = int(in0) + int(in1); a_hist[t] = a
    return 1 - err, size, a_hist

# ---------- arrays reused everywhere ----------
yr_all   = df_encoded['year'].values
y_all    = df_encoded['default'].values
p_all    = lgb_model.predict_proba(df_encoded[X_cols])[:, 1]
month_all = df['t_month'].values
cal_scores = true_label_scores(p_calib, y_calib.values)
te = test_mask.values
p_te, y_te, yr_te, m_te = p_all[te], y_all[te], yr_all[te], month_all[te]
print("Test rows chronological:", bool(np.all(np.diff(m_te) >= 0)), "| N test =", te.sum())

# Delayed feedback arrives as ONE BATCH per month: every loan of month t-d is revealed at the first loan of month t, so a per-loan
# step size gamma moves alpha_t by roughly gamma * (loans per month) in a single jump.  A gamma that is fine for ~170 loans/month is
# unstable for 2,500.  Delayed-feedback grids are therefore set from the batch gain  gamma * N_month  in {0.01, 0.1, 0.25}.
NPM = float(pd.Series(m_te).value_counts().mean())
_r1 = lambda x: float(f"{x:.1g}")
G_DELAY    = [_r1(x) for x in np.array((0.01, 0.1, 0.25)) / NPM]
G_DELAY_WF = [_r1(x) for x in np.array((0.003, 0.01, 0.03, 0.1, 0.25, 0.5)) / NPM]
print(f"Loans per test month ~ {NPM:.0f}; delayed-feedback step sizes (batch gain 0.01 / 0.1 / 0.25): {G_DELAY}")


## 11. ACI on Chronologically Ordered Data
Because the data are sorted by `issue_d`, the "online recovery" plot has a real time axis.

**How to read ACI's result.** ACI's long-run coverage holds for *any* sequence, so ≈90 % overall coverage is close to true by construction and is **not** evidence that the underlying model is well calibrated. The informative outputs are what ACI *pays* for it: larger prediction sets, how far $\alpha_t$ has to move, and how sensitive the result is to $\gamma$.

> **Caveat.** "Holds for any sequence" is true for ACI *as published* (α_t unclipped; α_t ≤ 0 ⇒ full set, α_t ≥ 1 ⇒ empty set) **with immediate feedback**. The `aci()` helper clips α_t to [0.01, 0.99] and the guarantee is then lost: at large γ the clipped variant can miss the published bound. §11b runs the unclipped algorithm and checks the bound numerically. Under delayed labels no such guarantee exists either way.

In [ ]:
alpha = 0.10
q_static = conf_q(cal_scores, alpha)
S_static = make_sets(p_te, q_static)
st = evaluate(S_static, y_te)

cov_a, size_a, a_hist = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=0.015)

# Guard against the two quantile / ACI implementations drifting apart.
assert np.isclose(q_static, q_hat), 'Section 4 threshold and conf_q disagree'
assert np.array_equal(cov_a, aci_covs), 'Section 6 ACI loop and aci() helper disagree'
assert np.array_equal(size_a, aci_sizes), 'Section 6 ACI set sizes and aci() helper disagree'

rows = []
for yr in (2013, 2014, 2015):
    m = yr_te == yr
    rows.append(dict(Year=yr,
        Static_cov=evaluate(S_static[m], y_te[m])['cov'], Static_size=S_static[m].sum(1).mean(),
        ACI_cov=cov_a[m].mean(), ACI_size=size_a[m].mean(),
        alpha_t_min=a_hist[m].min(), alpha_t_max=a_hist[m].max()))
rows.append(dict(Year='All', Static_cov=st['cov'], Static_size=st['size'], ACI_cov=cov_a.mean(),
                 ACI_size=size_a.mean(), alpha_t_min=a_hist.min(), alpha_t_max=a_hist.max()))
print("Static vs chronological ACI (alpha=0.10, gamma=0.015)")
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"\nalpha_t: mean={a_hist.mean():.3f}  std={a_hist.std():.3f}  "
      f"share of steps clipped at 0.01/0.99: {np.mean((a_hist<=0.01)|(a_hist>=0.99)):.2%}")

print("\nSensitivity to gamma (chronological, delay=0):")
g_rows = []
for g in (0.0005, 0.001, 0.005, 0.015, 0.05, 0.1):
    c, s_, ah = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g)
    g_rows.append(dict(gamma=g, coverage=c.mean(), cov_2015=c[yr_te==2015].mean(),
                       size=s_.mean(), alpha_t_std=ah.std(), alpha_t_min=ah.min()))
print(pd.DataFrame(g_rows).round(4).to_string(index=False))

dates_te = df.loc[te, 'issue_d'].values
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
mdf = pd.DataFrame({'d': pd.to_datetime(dates_te), 'aci': cov_a, 'static': S_static[np.arange(len(y_te)), y_te].astype(float), 'a': a_hist}) \
        .groupby('d').agg(aci=('aci', 'mean'), static=('static', 'mean'), a_mean=('a', 'mean'), a_lo=('a', 'min'), a_hi=('a', 'max'))
ax[0].plot(mdf.index, mdf['aci'], color='teal', marker='o', ms=3, label='ACI, monthly coverage')
ax[0].plot(mdf.index, mdf['static'], color='darkred', marker='s', ms=3, label='Static, monthly coverage')
ax[0].axhline(0.9, ls='--', c='k'); ax[0].set_title('Coverage by issue month (one point per month)', fontweight='bold'); ax[0].legend()
ax[1].fill_between(mdf.index, mdf['a_lo'], mdf['a_hi'], color='purple', alpha=0.15, label='within-month min-max')
ax[1].plot(mdf.index, mdf['a_mean'], color='purple', marker='o', ms=3, label='monthly mean'); ax[1].axhline(alpha, ls='--', c='k'); ax[1].legend()
ax[1].set_title(r'ACI working level $\alpha_t$ (target 0.10)', fontweight='bold'); plt.tight_layout(); save_fig('11_rolling_coverage_and_alpha_t'); plt.show()

## 11b. What ACI actually guarantees, and how order-sensitive it is
**(a) Unclipped ACI vs. the published bound.** Gibbs & Candès (2021) prove $\left|\tfrac1T\sum_t \mathrm{err}_t-\alpha\right|\le\frac{\max(\alpha_1,1-\alpha_1)+\gamma}{\gamma T}$ for *any* sequence — but only if $\alpha_t$ is **not clipped** (values $\le 0$ give the full set, $\ge 1$ the empty set). The `aci()` helper of §10 clips $\alpha_t$ to $[0.01,0.99]$, so at $\alpha_t=0.01$ the set is still finite (rank ⌈(n+1)·0.99⌉) and errors cannot be forced to zero. `aci_exact()` below is the published recursion; the bound is **asserted** for immediate feedback. With delayed labels no such bound exists for either version (§12).

**(b) Ordering.** `issue_d` is monthly, so the order of loans *within* a month is arbitrary, yet ACI is order-dependent. We re-run ACI over 30 random within-month orderings.

In [ ]:
def aci_exact(p, y, month, cal_scores, alpha=0.10, gamma=0.015, delay=0):
    # ACI as published: alpha_t unclipped; k <= 0 -> empty set (q = -inf), k > n -> full set (q = +inf).
    s = np.sort(cal_scores); n = len(s); T = len(y)
    a = alpha; err = np.zeros(T); size = np.zeros(T); a_hist = np.zeros(T); ptr = 0
    for t in range(T):
        while ptr < t and month[ptr] + delay <= month[t]:
            a = a + gamma * (alpha - err[ptr]); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a)))
        q = -np.inf if k <= 0 else (np.inf if k > n else s[k - 1])
        in0, in1 = p[t] <= q, (1 - p[t]) <= q
        err[t] = 1 - (in1 if y[t] == 1 else in0); size[t] = int(in0) + int(in1); a_hist[t] = a
    return 1 - err, size, a_hist

T_te = len(y_te)
rows = []
for g in (0.001, 0.005, 0.015, 0.05, 0.1, 0.3):
    c_c, s_c, ah_c = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g)              # clipped (used in Sections 6-19)
    c_e, s_e, ah_e = aci_exact(p_te, y_te, m_te, cal_scores, alpha, gamma=g)        # published
    bound = (max(alpha, 1 - alpha) + g) / (g * T_te) + max(alpha, 1 - alpha) / T_te   # + one term: the loop never applies the update after the last observation
    dev_e, dev_c = abs(c_e.mean() - (1 - alpha)), abs(c_c.mean() - (1 - alpha))
    assert dev_e <= bound + 1e-12, f"unclipped ACI violates the Gibbs-Candes bound at gamma={g}: {dev_e:.5f} > {bound:.5f}"
    rows.append(dict(gamma=g, cov_clipped=c_c.mean(), dev_clipped=dev_c, cov_unclipped=c_e.mean(), dev_unclipped=dev_e, GC_bound=bound,
                     clipped_within_bound=dev_c <= bound, size_clipped=s_c.mean(), size_unclipped=s_e.mean(),
                     alpha_t_min_unclipped=ah_e.min(), alpha_t_max_unclipped=ah_e.max(), share_steps_at_clip=np.mean((ah_c <= 0.01) | (ah_c >= 0.99))))
print("Immediate feedback: clipped (aci() helper) vs. published unclipped ACI; bound asserted for the unclipped one")
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nDelayed feedback (no guarantee for either version): coverage clipped vs. unclipped")
d_rows = []
for d in (6, 12, 24):
    for g in G_DELAY + [0.001]:                      # 0.001 x 2,500 loans/month = batch gain 2.5: kept as the unstable counter-example
        c_c, _, _ = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g, delay=d)
        c_e, s_e, ah_e = aci_exact(p_te, y_te, m_te, cal_scores, alpha, gamma=g, delay=d)
        d_rows.append(dict(delay=d, gamma=g, cov_clipped=c_c.mean(), cov_unclipped=c_e.mean(), cov2015_clipped=c_c[yr_te == 2015].mean(),
                           cov2015_unclipped=c_e[yr_te == 2015].mean(), size_unclipped=s_e.mean(), alpha_t_range=f"[{ah_e.min():.2f}, {ah_e.max():.2f}]"))
print(pd.DataFrame(d_rows).round(4).to_string(index=False))

# (b) within-month ordering robustness (uses the clipped helper, as in the rest of the notebook)
rng = np.random.default_rng(SEED); R = 30; o_rows = []
for r in range(R):
    o = np.lexsort((rng.random(T_te), m_te))              # primary key: month; random tie-break within month
    for label, kw in (('immediate, gamma=0.015', dict(gamma=0.015, delay=0)), (f'12-month delay, gamma={G_DELAY[1]:g}', dict(gamma=G_DELAY[1], delay=12))):
        c, s_, _ = aci(p_te[o], y_te[o], m_te[o], cal_scores, alpha, **kw)
        o_rows.append(dict(config=label, cov=c.mean(), cov_2015=c[yr_te[o] == 2015].mean(), size=s_.mean()))
df_o = pd.DataFrame(o_rows)
print(f"\nWithin-month ordering: mean +- sd [min, max] over {R} random orderings")
for label, g_ in df_o.groupby('config'):
    print(f"  {label}")
    for c in ('cov', 'cov_2015', 'size'):
        print(f"    {c:9s} {g_[c].mean():.4f} +- {g_[c].std():.4f}  [{g_[c].min():.4f}, {g_[c].max():.4f}]")
pd.DataFrame(rows).round(5).to_csv(f'{OUTDIR}/tables/11b_aci_clipped_vs_unclipped.csv', index=False)
pd.DataFrame(d_rows).round(5).to_csv(f'{OUTDIR}/tables/11b_aci_delay_clipped_vs_unclipped.csv', index=False)
df_o.round(5).to_csv(f'{OUTDIR}/tables/11b_aci_within_month_ordering.csv', index=False)

## 12. Delayed Label Feedback
Real defaults take 12–36 months to mature, so ACI cannot use a loan's outcome at issue time. Here the label of a loan issued in month $m$ is revealed only in month $m + d$, and $\alpha_t$ is updated only from revealed labels. We also run a crude *lagged recalibration* baseline: at test year $Y$, recalibrate the threshold on year $Y-1$ (labels assumed fully known).

### Step-size scale under delayed (batch) feedback
With a feedback delay, all labels of month *t − d* arrive together. With about 2,500 loans per month a per-loan step size γ therefore produces a single month-end jump of roughly **γ × N_month** in the working level α_t, with no feedback in between. The delayed grids below are set from this *batch gain* (γ × N_month ∈ {0.01, 0.1, 0.25}) rather than from per-loan values tuned for a much smaller monthly volume; the unscaled γ = 0.001 (batch gain ≈ 2.5) is kept as a counter-example. Coverage collapses there because of this scale, not because delayed ACI is inherently useless. The immediate-feedback (delay 0) rows are unaffected.


In [ ]:
d_rows = []
for d in (0, 6, 12, 24, 36):
    for g in ((0.0005, 0.001, 0.003, 0.015) if d == 0 else G_DELAY + [0.001]):   # delay > 0: batch-gain-scaled grid (+ the unscaled 0.001 for contrast)
        c, s_, ah = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g, delay=d)
        d_rows.append(dict(delay_months=d, gamma=g, coverage=c.mean(), cov_2013=c[yr_te==2013].mean(),
                           cov_2014=c[yr_te==2014].mean(), cov_2015=c[yr_te==2015].mean(),
                           size=s_.mean(), alpha_t_min=ah.min(), alpha_t_max=ah.max()))
dd = pd.DataFrame(d_rows)
print("ACI with delayed feedback: delay x gamma grid  (delay=36 => no label is revealed inside the test window => static)")
print(dd.round(4).to_string(index=False))

print("\nLagged recalibration baseline (threshold from year Y-1, model fixed):")
lag = []
for yr in (2013, 2014, 2015):
    prev = yr_all == yr - 1
    q_prev = conf_q(true_label_scores(p_all[prev], y_all[prev]), alpha)
    m = yr_te == yr
    e = evaluate(make_sets(p_te[m], q_prev), y_te[m])
    lag.append(dict(Year=yr, q_static=q_static, q_lagged=q_prev, cov=e['cov'], cov_default=e['cov_d'], size=e['size']))
print(pd.DataFrame(lag).round(4).to_string(index=False))


## 12b. Recalibration baselines, quantile tracking and local coverage
**Why.** (1) *Consistency.* §11–12 keep the full 2012 set as a matured calibration reference while delaying test labels by 6–36 months. A loan issued in late 2012 with a 12–36-month default window is not labelled at the start of 2013, so those runs are optimistic; here the recalibration baselines use **only labels that could have matured** (a loan from month $m'$ is usable in month $m$ iff $m'+d<m$), falling back to the static threshold when fewer than 150 labelled loans qualify (the share is reported). (2) *Baselines a reviewer asks for.* Sliding-window recalibration of the threshold, and quantile tracking $q_{t+1}=q_t+\eta(\mathrm{err}_t-\alpha)$ — the P-term of conformal PID control (Angelopoulos, Candès & Tibshirani, 2023), which works in score space and needs no clipping. (3) *Local coverage.* Long-run coverage can be met while local coverage swings; the rolling-window minimum and the share of windows within ±3 pp of 90 % are reported. (4) *Hindsight benchmark.* The smallest symmetric-cutoff set that would have given 90 % in each year, i.e. what validity costs when the shift is known in advance.

The frozen model is used throughout; delay is in months and delay 0 means "labels of all earlier months" for the recalibration rows (ACI and the tracker keep the notebook's convention that earlier loans in the same month are revealed).

In [ ]:
# ---------- §12b: quantile tracker (P-term of conformal PID), sliding-window recalibration, local-coverage metrics ----------
def quantile_tracker(p, y, month, q0, alpha=0.10, eta=0.01, delay=0):
    """Quantile tracking in score space (the P-term of conformal PID control, Angelopoulos, Candes & Tibshirani 2023):
    q_{t+1} = q_t + eta * (err_t - alpha).  Same label-reveal rule as aci().  q is not clipped (q >= 1 -> full set, q < 0 -> empty set)."""
    T = len(y); q = float(q0); err = np.zeros(T); size = np.zeros(T); qh = np.zeros(T); ptr = 0
    for t in range(T):
        while ptr < t and month[ptr] + delay <= month[t]:
            q += eta * (err[ptr] - alpha); ptr += 1
        in0, in1 = p[t] <= q, (1 - p[t]) <= q
        err[t] = 1 - (in1 if y[t] == 1 else in0); size[t] = int(in0) + int(in1); qh[t] = q
    return 1 - err, size, qh

def rolling_recal(p_cal, y_cal, m_cal, p, y, m, alpha=0.10, window=12, delay=0, n_min=150, q_fallback=None):
    """Frozen model, sliding-window recalibration.  The label of a loan issued in month m' is usable for predictions in month m iff
    m' + delay < m (so delay=0 means 'labels of all earlier months').  Only the last `window` issue months are used; the pool is the
    2012 calibration rows plus already-revealed stream rows.  If fewer than n_min labelled loans qualify, the full-2012 static threshold
    is used as a fallback (flagged).  Unlike the ACI runs of Section 12 (which keep the whole 2012 pool as a matured reference),
    this uses ONLY labels that could actually have matured, so with delay >= 12 the first test months fall back to the static threshold."""
    s_all = np.r_[true_label_scores(p_cal, y_cal), true_label_scores(p, y)]; m_all = np.r_[m_cal, m]
    q_fb = conf_q(true_label_scores(p_cal, y_cal), alpha) if q_fallback is None else q_fallback
    qm, fm = {}, {}
    for mm in np.unique(m):
        hi = mm - delay - 1; lo = hi - window + 1
        pool = s_all[(m_all >= lo) & (m_all <= hi)]
        fm[mm] = len(pool) < n_min; qm[mm] = q_fb if fm[mm] else conf_q(pool, alpha)
    q_t = np.array([qm[x] for x in m]); fb = np.array([fm[x] for x in m])
    in0, in1 = p <= q_t, (1 - p) <= q_t
    return np.where(y == 1, in1, in0).astype(float), (in0.astype(int) + in1.astype(int)).astype(float), q_t, fb

def local_cov_stats(cov, window=300, target=0.90):
    """Long-run coverage can be met while local coverage swings; report the rolling-window spread as well."""
    r = pd.Series(cov).rolling(window).mean().dropna().values
    return dict(min_roll_cov=r.min(), max_abs_dev=np.abs(r - target).max(), share_within_3pp=np.mean(np.abs(r - target) <= 0.03))

def oracle_q(scores, alpha):
    """Hindsight threshold: the ceil(n(1-alpha))-th smallest score of the very year it is applied to (coverage >= 1-alpha by construction)."""
    s = np.sort(scores); return s[int(np.ceil(len(s) * (1 - alpha))) - 1]

cal_m = month_all[calib_mask.values]; y_cal_arr2 = y_calib.values
s_te = true_label_scores(p_te, y_te); T_te = len(y_te)

# ---- identity / bound checks ----
c_id, _, _, fb_id = rolling_recal(p_calib, y_cal_arr2, cal_m, p_te, y_te, m_te, alpha, window=10_000, delay=10_000)
assert fb_id.all() and np.array_equal(c_id, S_static[np.arange(T_te), y_te].astype(float)), "empty pool must reproduce static conformal"
for eta_ in (0.005, 0.02, 0.1):                      # deterministic bound for the P-term with immediate feedback (assumes 0 < q0 < 1)
    c_, _, qh_ = quantile_tracker(p_te, y_te, m_te, q_static, alpha, eta_, delay=0)
    q_next = qh_[-1] + eta_ * ((1 - c_[-1]) - alpha)          # apply the final update so the telescoping identity is exact
    assert abs(q_next - q_static) <= 1 + eta_ + 1e-12
    assert abs(c_.mean() - (1 - alpha)) <= (1 + eta_) / (eta_ * T_te) + 1e-12, f"quantile tracker violates its own bound at eta={eta_}"
print("Checks passed: empty-pool recalibration == static; quantile-tracker coverage within (1+eta)/(eta*T) of 0.90 (immediate feedback).")

# ---- comparison grid ----
rows = []
def add_row(method, delay, cov, size, fb=None):
    r = dict(method=method, delay=delay, cov_all=cov.mean(), cov_2013=cov[yr_te == 2013].mean(), cov_2014=cov[yr_te == 2014].mean(),
             cov_2015=cov[yr_te == 2015].mean(), size_all=size.mean(), size_2015=size[yr_te == 2015].mean())
    r.update(local_cov_stats(cov))
    if fb is not None: r['share_fallback'] = fb.mean()
    rows.append(r)

add_row('Static (2012 threshold)', '-', S_static[np.arange(T_te), y_te].astype(float), S_static.sum(1).astype(float))
cov_or, size_or = np.zeros(T_te), np.zeros(T_te)
for yr in (2013, 2014, 2015):
    m_ = yr_te == yr; Sy = make_sets(p_te[m_], oracle_q(s_te[m_], alpha))
    cov_or[m_] = Sy[np.arange(m_.sum()), y_te[m_]]; size_or[m_] = Sy.sum(1)
add_row('Oracle per-year threshold (hindsight)', '-', cov_or, size_or)
for d in (0, 6, 12):
    for W in (6, 12):
        c_, s_, _, fb_ = rolling_recal(p_calib, y_cal_arr2, cal_m, p_te, y_te, m_te, alpha, window=W, delay=d)
        add_row(f'Rolling recalibration, W={W} mo', d, c_, s_, fb_)
    for g in ((0.005, 0.015) if d == 0 else (G_DELAY[1], G_DELAY[2])):
        c_, s_, _ = aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g, delay=d); add_row(f'ACI, gamma={g}', d, c_, s_)
    for eta_ in ((0.005, 0.02) if d == 0 else (G_DELAY[1], G_DELAY[2])):
        c_, s_, _ = quantile_tracker(p_te, y_te, m_te, q_static, alpha, eta_, delay=d); add_row(f'Quantile tracker, eta={eta_}', d, c_, s_)
df_grid = pd.DataFrame(rows)
print("Adaptive / recalibrating methods under label delay (frozen model, alpha=0.10; delay in months; local metrics use 300-loan windows):")
print(df_grid.round(4).to_string(index=False))
print("\nRead against the hindsight-threshold row: the smallest single symmetric threshold per year that delivers >= 90 % in that year (a reference for set size, not a bound on causal methods).")
print("ACI / tracker rows at delay 0 are the (unrealistic) immediate-feedback case; rolling recalibration uses only labels that have matured. Step sizes (gamma, eta) are NOT tuned here: see 16c for walk-forward gamma.")

# ---- coverage-size frontier ----
fig, axs = plt.subplots(1, 2, figsize=(14, 5.2)); qg = np.linspace(0.5, 0.995, 200)
mk = {'Static': 'k*', 'Oracle': 'g*', 'Rolling': 's', 'ACI': 'o', 'Quantile': '^'}; col = {'0': 'tab:blue', '6': 'tab:orange', '12': 'tab:red', '-': 'k'}
for ax_, sel, cc, sc_, title in ((axs[0], np.ones(T_te, bool), 'cov_all', 'size_all', 'pooled 2013-2015'), (axs[1], yr_te == 2015, 'cov_2015', 'size_2015', '2015 only')):
    ax_.plot([np.mean(p_te[sel] <= q) + np.mean(1 - p_te[sel] <= q) for q in qg], [np.mean(s_te[sel] <= q) for q in qg], 'k-', lw=1, alpha=.6,
             label='any fixed symmetric threshold (hindsight)')
    for _, r in df_grid.iterrows():
        fam = next(k for k in mk if r.method.startswith(k)); c_ = col[str(r.delay)]
        ax_.scatter(r[sc_], r[cc], marker=mk[fam][-1], s=90, color='g' if fam == 'Oracle' else c_, edgecolor='k', zorder=3)
    ax_.axhline(0.9, ls='--', c='grey'); ax_.set_xlabel('mean set size'); ax_.set_ylabel('coverage'); ax_.set_title(title, fontweight='bold')
from matplotlib.lines import Line2D
axs[1].legend(handles=[Line2D([], [], marker='s', ls='', color='grey', label='rolling recal.'), Line2D([], [], marker='o', ls='', color='grey', label='ACI'),
                       Line2D([], [], marker='^', ls='', color='grey', label='quantile tracker'), Line2D([], [], marker='*', ls='', color='k', label='static / oracle'),
                       Line2D([], [], marker='o', ls='', color='tab:blue', label='delay 0'), Line2D([], [], marker='o', ls='', color='tab:orange', label='delay 6'),
                       Line2D([], [], marker='o', ls='', color='tab:red', label='delay 12')], loc='lower right', fontsize=8)
plt.tight_layout(); save_fig('12b_coverage_size_frontier')
plt.show()
df_grid.round(5).to_csv(f'{OUTDIR}/tables/12b_recalibration_tracker_grid.csv', index=False)

## 13. Class-Conditional (Mondrian) Conformal
Marginal coverage can hide the class that matters. A defaulter's score $1-\hat p(1|x)$ is typically much larger than a non-defaulter's, so one shared threshold over-covers the majority class and under-covers defaults. Mondrian conformal calibrates a **separate threshold per true class**, giving a class-conditional guarantee (under exchangeability).

In [ ]:
def mondrian_q(p_cal, y_cal, alpha):
    s0 = p_cal[y_cal == 0]            # score of label 0 = p
    s1 = 1 - p_cal[y_cal == 1]        # score of label 1 = 1-p
    return conf_q(s0, alpha), conf_q(s1, alpha)

q0, q1 = mondrian_q(p_calib, y_calib.values, alpha)
S_mond = make_sets(p_te, q0, q1)
rows = []
for name, S in (("Static (marginal)", S_static), ("Mondrian", S_mond)):
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
        e = evaluate(S[m], y_te[m])
        rows.append(dict(Method=name, Year=yr, Cov=e['cov'], Cov_nonDefault=e['cov_nd'],
                         Cov_Default=e['cov_d'], SetSize=e['size'],
                         Empty=np.mean(S[m].sum(1) == 0)))
df_mond = pd.DataFrame(rows)
print(df_mond.round(4).to_string(index=False))

# Mondrian targets ~90% coverage for BOTH classes. Under shift it relocates the failure rather than fixing it: non-default coverage itself can fall below 90%.
mond_nd_fail = df_mond[(df_mond.Method == 'Mondrian') & (df_mond.Year != 'All') & (df_mond.Cov_nonDefault < 0.90)]
if len(mond_nd_fail):
    print("\nNote: Mondrian's non-default-class coverage also falls below the 90% target under shift:")
    for _, r in mond_nd_fail.iterrows():
        print(f"  {int(r.Year)}: non-default coverage {r.Cov_nonDefault:.1%} (target 90%), "
              f"while default coverage over-shoots to {r.Cov_Default:.1%}.")
    print("  -> Mondrian does not restore validity under shift; it shifts the miscoverage from")
    print("     the minority (default) class onto the majority (non-default) class.")

df_mond.round(4).to_csv(f'{OUTDIR}/tables/13_mondrian_vs_static.csv', index=False)


## 13b. Conditional coverage beyond the class
Marginal and class-conditional coverage can both look fine while a risk band or a borrower segment is badly under-covered. This section reports coverage for every level of seven slice families (predicted-risk decile, FICO / loan-size / `int_rate` terciles, home ownership, verification, purpose; slices with n ≥ 100), the **worst slice per family**, and a **bin-conditional (Mondrian-by-$\hat p$) conformal** variant that calibrates a threshold inside each of five predicted-risk bins. Slice-level intervals are wide; treat the worst-slice column as a screening statistic, not a test.

In [ ]:
# ---------- §13b conditional coverage: slices, worst-slice coverage, bin-conditional (Mondrian-by-p-hat) conformal ----------
te_df = df.loc[te].reset_index(drop=True)
def _tercile(x, name):
    return pd.qcut(x, 3, labels=[f'{name}:low', f'{name}:mid', f'{name}:high']).astype(str)

slices = {
    'risk decile (p_hat)': pd.Series(pd.qcut(p_te, 10, labels=False, duplicates='drop')).map(lambda k: f'D{int(k):02d}'),
    'FICO tercile':        pd.Series(_tercile(te_df['fico_score'], 'FICO')),
    'loan-size tercile':   pd.Series(_tercile(te_df['loan_amnt'], 'amount')),
    'int_rate tercile':    pd.Series(_tercile(te_df['int_rate'], 'rate')),
    'home_ownership':      te_df['home_ownership'].astype(str),
    'verification':        te_df['verification_status'].astype(str),
    'purpose':             te_df['purpose'].astype(str),
}

def bin_mondrian_sets(p_cal, y_cal, p, alpha, n_bins=5):
    """Conformal thresholds calibrated separately inside bins of the predicted risk p_hat (a taxonomy that depends on x only),
    which targets coverage *conditional on the risk bin* rather than on the class. Valid under exchangeability."""
    edges = np.quantile(p_cal, np.linspace(0, 1, n_bins + 1)[1:-1])
    b_cal, b_te = np.searchsorted(edges, p_cal), np.searchsorted(edges, p)
    sc = true_label_scores(p_cal, y_cal)
    qb = np.array([conf_q(sc[b_cal == b], alpha) for b in range(n_bins)])
    return make_sets(p, qb[b_te])

S_bin = bin_mondrian_sets(p_calib, y_calib.values, p_te, alpha, n_bins=5)
methods_cc = {'Static (marginal)': S_static, 'Mondrian (class)': S_mond, 'Bin-Mondrian (5 p_hat bins)': S_bin}

def slice_cov_table(S, mask=None, min_n=100):
    hit = S[np.arange(len(y_te)), y_te]; base = np.ones(len(y_te), bool) if mask is None else mask
    out = []
    for fam, ser in slices.items():
        for lvl in sorted(ser.unique()):
            mm = base & (ser.values == lvl)
            if mm.sum() < min_n: continue
            lo, hi = proportion_confint(int(hit[mm].sum()), int(mm.sum()), alpha=0.05, method='wilson')
            out.append(dict(family=fam, level=lvl, n=int(mm.sum()), coverage=hit[mm].mean(), ci_lo=lo, ci_hi=hi))
    return pd.DataFrame(out)

tabs = {name: slice_cov_table(S) for name, S in methods_cc.items()}
summ = []
for name, tb in tabs.items():
    for fam, g in tb.groupby('family'):
        w = g.loc[g.coverage.idxmin()]
        summ.append(dict(method=name, family=fam, worst_level=w.level, worst_cov=w.coverage, worst_gap_pp=100 * (w.coverage - (1 - alpha)),
                         best_cov=g.coverage.max(), n_slices=len(g)))
df_slice_summary = pd.DataFrame(summ)
print("Worst-slice coverage (pooled 2013-2015, slices with n >= 100, alpha = 0.10):")
print(df_slice_summary.pivot_table(index='family', columns='method', values='worst_cov').round(3).to_string())
print("\nWorst slice per method and family:")
print(df_slice_summary[['method', 'family', 'worst_level', 'worst_cov']].round(3).to_string(index=False))

dec = pd.concat({n: t[t.family == 'risk decile (p_hat)'].set_index('level')['coverage'] for n, t in tabs.items()}, axis=1)
print("\nCoverage by predicted-risk decile (pooled 2013-2015):"); print(dec.round(3).to_string())
_w = df_slice_summary[df_slice_summary.family == 'risk decile (p_hat)'].set_index('method')['worst_cov']
print("\nWorst risk-decile coverage (target %.2f): " % (1 - alpha) + ", ".join(f"{k} {v:.3f}" for k, v in _w.items()))
print("Bin-Mondrian is valid within each of its own p_hat bins under exchangeability; it gives no guarantee for slices that are not aligned with p_hat, and none under drift.")
print("Set sizes:", {n: round(float(S.sum(1).mean()), 3) for n, S in methods_cc.items()})

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for name in methods_cc: ax.plot(range(len(dec)), dec[name].values, marker='o', label=name)
ax.axhline(1 - alpha, ls='--', c='k'); ax.set_xticks(range(len(dec))); ax.set_xticklabels(dec.index, rotation=45)
ax.set_ylabel('coverage'); ax.set_title('Coverage conditional on predicted-risk decile (2013-2015)', fontweight='bold'); ax.legend()
plt.tight_layout(); save_fig('13b_conditional_coverage'); plt.show()
pd.concat([t.assign(method=n) for n, t in tabs.items()]).round(4).to_csv(f'{OUTDIR}/tables/13b_slice_coverage.csv', index=False)
df_slice_summary.round(4).to_csv(f'{OUTDIR}/tables/13b_worst_slice_summary.csv', index=False)

## 14. Control Experiment: Same Pipeline on an I.I.D. Split
To attribute the coverage failure to temporal shift, we run the **identical** pipeline (same model, hyperparameters, split sizes as in Section 2, same $\alpha$) on random splits of the pooled 2007–2015 data, over 30 seeds.

In [ ]:
n_tr, n_ca = int(train_mask.sum()), int(calib_mask.sum())
Xf, yf = df_encoded[X_cols], df_encoded['default'].values
rows = []
for seed in range(30):
    perm = np.random.default_rng(seed).permutation(len(df_encoded))
    i_tr, i_ca, i_te = perm[:n_tr], perm[n_tr:n_tr+n_ca], perm[n_tr+n_ca:]
    mdl = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)
    mdl.fit(Xf.iloc[i_tr], yf[i_tr])
    pc, pt = mdl.predict_proba(Xf.iloc[i_ca])[:, 1], mdl.predict_proba(Xf.iloc[i_te])[:, 1]
    q = conf_q(true_label_scores(pc, yf[i_ca]), alpha)
    e = evaluate(make_sets(pt, q), yf[i_te])
    a0, a1 = mondrian_q(pc, yf[i_ca], alpha)
    em = evaluate(make_sets(pt, a0, a1), yf[i_te])
    rows.append(dict(seed=seed, cov=e['cov'], cov_default=e['cov_d'], size=e['size'],
                     mond_cov=em['cov'], mond_cov_default=em['cov_d'], mond_size=em['size']))
iid = pd.DataFrame(rows)
print("I.I.D. control (30 random splits): mean ± sd, [min, max]")
for c in iid.columns[1:]:
    print(f"  {c:18s} {iid[c].mean():.4f} ± {iid[c].std():.4f}   [{iid[c].min():.4f}, {iid[c].max():.4f}]")
print(f"\nTemporal split, static:   coverage {st['cov']:.4f}, default coverage {st['cov_d']:.4f}, size {st['size']:.3f}")


## 15. Diagnosing the Shift
Three checks separate *covariate* shift ($P(X)$ changes) from *label / concept* drift ($P(Y|X)$ changes):
1. **Calibration-in-the-large:** mean predicted risk vs. realised default rate per year.
2. **Domain classifier:** can a model tell training-period rows (2007–2011) from year-$Y$ rows using only $X$? (5-fold CV AUC; 0.5 = no covariate shift.)
3. **Covariate-shift-weighted conformal** (Tibshirani et al., 2019): if covariate shift were the problem, reweighting the 2012 calibration scores by the density ratio toward year $Y$ should restore coverage.

In [ ]:
def domain_auc(mask_a, mask_b):
    Xd = pd.concat([df_encoded.loc[mask_a, X_cols], df_encoded.loc[mask_b, X_cols]])
    yd = np.r_[np.zeros(mask_a.sum()), np.ones(mask_b.sum())]
    clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=0, verbose=-1)
    pr = cross_val_predict(clf, Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=0), method='predict_proba')[:, 1]
    return roc_auc_score(yd, pr)

def weighted_q(scores, w, w_test, alpha):
    """Single shared threshold for a whole batch, using one aggregate w_test
    (kept for comparison -- this is the coarse version originally used here)."""
    o = np.argsort(scores); s, w = scores[o], w[o]
    cum = np.cumsum(w) / (w.sum() + w_test)
    idx = np.searchsorted(cum, 1 - alpha)
    return np.inf if idx >= len(s) else s[idx]

def weighted_q_per_point(scores, w_cal, w_test_vec, alpha):
    """Weighted conformal of Tibshirani et al. (2019): each test point i gets its OWN threshold, obtained by renormalising the
    calibration weights together with that point's own weight w_test_vec[i] (the augmented 'infinite' score at the test point).
    Vectorised across test points via searchsorted."""
    o = np.argsort(scores)
    s, w = scores[o], w_cal[o]
    cum = np.cumsum(w)
    total = w.sum()
    target = (1 - alpha) * (total + w_test_vec)          # one target per test point
    idx = np.searchsorted(cum, target)
    return np.where(idx >= len(s), np.inf, s[np.minimum(idx, len(s) - 1)])

rows = []
for yr in range(2012, 2016):
    m = yr_all == yr
    # density ratio calib(2012) -> year yr via a domain classifier
    if yr == 2012:
        cov_w_coarse, cov_w_fine, mean_q_fine = np.nan, np.nan, np.nan
    else:
        Xd = pd.concat([df_encoded.loc[calib_mask, X_cols], df_encoded.loc[m, X_cols]])
        yd = np.r_[np.zeros(calib_mask.sum()), np.ones(m.sum())]
        clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=0, verbose=-1).fit(Xd, yd)
        pc_cal = np.clip(clf.predict_proba(df_encoded.loc[calib_mask, X_cols])[:, 1], 1e-3, 1 - 1e-3)
        pc_test = np.clip(clf.predict_proba(df_encoded.loc[m, X_cols])[:, 1], 1e-3, 1 - 1e-3)
        scale = calib_mask.sum() / m.sum()
        w_cal = pc_cal / (1 - pc_cal) * scale                 # calibration-point weights
        w_test_vec = pc_test / (1 - pc_test) * scale          # ONE weight PER test point

        # coarse (original) version: single shared threshold using the mean test weight
        wq_coarse = weighted_q(cal_scores, w_cal, w_test_vec.mean(), alpha)
        cov_w_coarse = evaluate(make_sets(p_all[m], wq_coarse), y_all[m])['cov']

        # fine (fixed) version: per-test-point threshold
        q_fine = weighted_q_per_point(cal_scores, w_cal, w_test_vec, alpha)
        p_m = p_all[m]
        in0, in1 = p_m <= q_fine, (1 - p_m) <= q_fine
        y_m = y_all[m]
        covered_fine = np.where(y_m == 1, in1, in0)
        cov_w_fine = covered_fine.mean()
        mean_q_fine = np.mean(q_fine[np.isfinite(q_fine)])

    rows.append(dict(Year=yr, mean_pred_risk=p_all[m].mean(), actual_default=y_all[m].mean(),
        gap=y_all[m].mean() - p_all[m].mean(), mean_FICO=df.loc[m, 'fico_score'].mean(),
        mean_DTI=df.loc[m, 'dti'].mean(), AUC_domain_vs_train=domain_auc(train_mask.values, m),
        AUC_model=roc_auc_score(y_all[m], p_all[m]),
        static_cov=evaluate(make_sets(p_all[m], q_static), y_all[m])['cov'],
        cov_shift_weighted_coarse=cov_w_coarse, cov_shift_weighted_perpoint=cov_w_fine))
diag = pd.DataFrame(rows)
print(diag.round(4).to_string(index=False))
print("\ncov_shift_weighted_coarse = single year-averaged test weight (coarse variant, kept for comparison)")
print("cov_shift_weighted_perpoint = one weight per test point (Tibshirani et al. 2019 form)")

rate_by_year = df.groupby('year')['int_rate'].mean()
print("\nMean int_rate by year:", rate_by_year.round(2).to_dict())

diag.round(4).to_csv(f'{OUTDIR}/tables/15_shift_diagnosis.csv', index=False)

# calibration curves
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for yr, col in ((2012, 'tab:blue'), (2013, 'tab:green'), (2014, 'tab:orange'), (2015, 'tab:red')):
    m = yr_all == yr
    b = pd.qcut(p_all[m], 10, duplicates='drop')
    g = pd.DataFrame({'p': p_all[m], 'y': y_all[m]}).groupby(b, observed=True).mean()
    ax[0].plot(g.p, g.y, marker='o', color=col, label=str(yr))
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_xlabel('mean predicted risk (decile)'); ax[0].set_ylabel('observed default rate')
ax[0].set_title('Reliability by year: the curve lifts (label drift)', fontweight='bold'); ax[0].legend()
ax[1].bar(diag.Year - 0.2, diag.mean_pred_risk, 0.4, label='mean predicted'); ax[1].bar(diag.Year + 0.2, diag.actual_default, 0.4, label='actual')
ax[1].set_title('Predicted vs. actual default rate', fontweight='bold'); ax[1].legend(); plt.tight_layout()
save_fig('15_shift_diagnosis')
plt.show()

## 15b. Shift Diagnosis, Continued
Section 15 measured domain-classifier AUC against the *whole 2007–2011 training pool*, a number confounded by `int_rate` (the pool contains the 2008–09 rate spike), so it is **not** used as evidence. Three diagnostics replace it:
1. **Domain AUC against the calibration year (2012)** — the reference that actually matters for a conformal threshold calibrated on 2012 — cross-validated.
2. **Feature importance** of the domain classifiers, to test (not assert) the `int_rate` explanation.
3. **Cross-fitted covariate-shift weights.** Section 15 fit the density-ratio classifier on the same rows it then scored, which makes the weights over-dispersed. Here each row's weight comes from a fold that did not see it, and the effective sample size of the weighted calibration set is reported.

In [ ]:
def domain_fit(mask_a, mask_b, seed=0):
    """5-fold out-of-fold domain-classifier probabilities for A (label 0) vs B (label 1), plus gain importances."""
    Xd = pd.concat([df_encoded.loc[mask_a, X_cols], df_encoded.loc[mask_b, X_cols]])
    yd = np.r_[np.zeros(int(mask_a.sum())), np.ones(int(mask_b.sum()))]
    clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=0, verbose=-1,
                             importance_type='gain')
    oof = cross_val_predict(clf, Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                            method='predict_proba')[:, 1]
    clf.fit(Xd, yd)
    imp = pd.Series(clf.feature_importances_, index=X_cols, dtype=float)
    return yd, oof, imp / imp.sum()

# (1)+(2) what drives the train-pool vs 2012 separation that Section 15 flagged?
yd0, oof0, imp_tr = domain_fit(train_mask.values, calib_mask.values)
print(f"Domain AUC, train pool (2007-2011) vs 2012 (5-fold CV): {roc_auc_score(yd0, oof0):.4f}")
print("Top features (share of gain):", imp_tr.sort_values(ascending=False).head(4).round(3).to_dict())

rows = []
n_c = int(calib_mask.sum())
diag_i = diag.set_index('Year')
for yr in (2013, 2014, 2015):
    m = yr_all == yr
    yd, oof, imp = domain_fit(calib_mask.values, m)              # reference = the calibration year
    pc = np.clip(oof[:n_c], 1e-3, 1 - 1e-3)                      # out-of-fold, calibration rows
    pt = np.clip(oof[n_c:], 1e-3, 1 - 1e-3)                      # out-of-fold, test-year rows
    scale = n_c / m.sum()
    w_cal, w_te = pc / (1 - pc) * scale, pt / (1 - pt) * scale
    q_cf = weighted_q_per_point(cal_scores, w_cal, w_te, alpha)  # per-point Tibshirani et al. (2019) threshold
    p_m, y_m = p_all[m], y_all[m]
    in0, in1 = p_m <= q_cf, (1 - p_m) <= q_cf
    ess = w_cal.sum() ** 2 / (w_cal ** 2).sum()
    rows.append(dict(
        Year=yr,
        AUC_domain_vs_calib_cv=roc_auc_score(yd, oof),
        top_feature=imp.idxmax(), top_feature_share=imp.max(),
        ESS_calib=ess, ESS_share=ess / n_c,
        static_cov=evaluate(make_sets(p_m, q_static), y_m)['cov'],
        cov_weighted_insample=diag_i.loc[yr, 'cov_shift_weighted_perpoint'],
        cov_weighted_crossfit=np.where(y_m == 1, in1, in0).mean()))
diag_b = pd.DataFrame(rows)
print("\nDomain AUC against the calibration year, and covariate-shift-weighted coverage (in-sample vs cross-fitted weights):")
print(diag_b.round(4).to_string(index=False))
print("\nHow to read: AUC_domain_vs_calib_cv ~ 0.5 means 2012 -> year Y shows little covariate shift; ESS_share far below 1 means the")
print("weighting is dominated by a few calibration points. Compare cov_weighted_crossfit with static_cov and with 0.90 before")
print("claiming covariate shift does (or does not) explain the 2015 shortfall.")
diag_b.round(4).to_csv(f'{OUTDIR}/tables/15b_domain_diagnostics_vs_calibration.csv', index=False)

## 15c. Is it prior (label) shift? Label-free diagnosis and adaptation
Conclusion 2 says the covariate/concept split is not exhaustive: a rising base rate with stable $P(X\mid Y)$ (**prior shift**) would look the same. It is directly testable, and — unlike ACI or recalibration — needs **no labels of the test period**: the class prior of the current portfolio can be estimated from unlabelled model scores (EM, Saerens et al., 2002; black-box shift estimation, Lipton et al., 2018) and fed into **label-shift-weighted conformal** (calibration point $i$ weighted by $\pi_{\text{test}}(y_i)/\pi_{\text{cal}}(y_i)$; test point adds mass $w(c)$ at $+\infty$ for candidate label $c$; Tibshirani et al., 2019; Podkopaev & Ramdas, 2021). The online version re-estimates the prior every month from the previous three months of unlabelled scores.

**Interpretation rule.** If the estimated-prior rows close most of the 2015 shortfall, prior shift is a sufficient explanation *and* a label-free fix exists. If the *oracle-prior* row also fails, the drift is in $P(Y\mid X)$ and no reweighting of class frequencies can help. The EM step assumes the model's probabilities are calibrated for the 2012 prior (checked by the calibration-in-the-large line printed below); BBSE does not.

In [ ]:
# ---------- §15c label-shift (prior-shift) diagnosis and label-shift-weighted conformal; uses ONLY unlabelled test covariates ----------
def em_prior(p, pi_ref, iters=1000, tol=1e-10):
    """Saerens-Latinne-Decaestecker (2002) EM estimate of P(Y=1) in an unlabelled batch, given p = model P(Y=1|x) that is calibrated for prior pi_ref."""
    pi = pi_ref
    for _ in range(iters):
        a, b = pi / pi_ref * p, (1 - pi) / (1 - pi_ref) * (1 - p)
        new = float(np.mean(a / (a + b)))
        if abs(new - pi) < tol: pi = new; break
        pi = new
    return float(np.clip(pi, 0.01, 0.99))

def bbse_prior(p_cal, y_cal, p_unl, thr=0.5):
    """Black-box shift estimation (Lipton et al. 2018), binary case = Rogan-Gladen: pi1 = (P_unl(yhat=1) - FPR) / (TPR - FPR), rates from the calibration set."""
    yh = p_cal >= thr; tpr, fpr = yh[y_cal == 1].mean(), yh[y_cal == 0].mean()
    return float(np.clip((np.mean(p_unl >= thr) - fpr) / (tpr - fpr), 0.01, 0.99))

def ls_thresholds(scores, y_cal, pi_te1, alpha):
    """Label-shift-weighted conformal thresholds (Podkopaev & Ramdas 2021 / Tibshirani et al. 2019 with w(y) = pi_test(y)/pi_cal(y)).
    Calibration point i has weight w(y_i); a test point with candidate label c adds mass w(c) at +inf. Returns (q for candidate 0, q for candidate 1)."""
    pi_c1 = float(np.mean(y_cal)); w_cls = np.array([(1 - pi_te1) / (1 - pi_c1), pi_te1 / pi_c1])
    w = w_cls[np.asarray(y_cal).astype(int)]; o = np.argsort(scores); s, ww = scores[o], w[o]; cum = np.cumsum(ww); tot = ww.sum()
    qs = []
    for c in (0, 1):
        idx = np.searchsorted(cum, (1 - alpha) * (tot + w_cls[c]))
        qs.append(np.inf if idx >= len(s) else s[idx])
    return qs[0], qs[1]

def ls_online(p_cal, y_cal, m_cal, p, y, m, alpha=0.10, method='em', trail=3, min_unl=300, oracle=None):
    """Causal, label-free adaptation. For each test month, the class prior is estimated from the (unlabelled) model scores of loans issued in the
    previous `trail` months and the label-shift-weighted thresholds are recomputed. `oracle` (dict month -> prior) replaces the estimate."""
    pi_c, pi_ref = float(np.mean(y_cal)), float(np.mean(p_cal)); cs = true_label_scores(p_cal, y_cal)
    p_pool, m_pool = np.r_[p_cal, p], np.r_[m_cal, m]; q0s, q1s, pis = {}, {}, {}
    for mm in np.unique(m):
        unl = p_pool[(m_pool >= mm - trail) & (m_pool <= mm - 1)]
        if oracle is not None: pi = oracle[mm]
        elif len(unl) < min_unl: pi = pi_c
        elif method == 'em': pi = em_prior(unl, pi_ref)
        else: pi = bbse_prior(p_cal, y_cal, unl)
        pis[mm] = pi; q0s[mm], q1s[mm] = ls_thresholds(cs, y_cal, pi, alpha)
    q0, q1, ph = (np.array([d_[x] for x in m]) for d_ in (q0s, q1s, pis))
    in0, in1 = p <= q0, (1 - p) <= q1
    return np.where(y == 1, in1, in0).astype(float), (in0.astype(int) + in1.astype(int)).astype(float), ph

cal_m = month_all[calib_mask.values]; y_cal_arr2 = y_calib.values; cs_ = true_label_scores(p_calib, y_cal_arr2)
# identity: unchanged prior -> plain split conformal
q0_id, q1_id = ls_thresholds(cs_, y_cal_arr2, float(y_cal_arr2.mean()), alpha)
assert np.isclose(q0_id, q_static) and np.isclose(q1_id, q_static), "unit weights must reproduce conf_q"

year_of_month = {int(mm): int(yr_te[m_te == mm][0]) for mm in np.unique(m_te)}
prior_by_year = {yr: float(y_te[yr_te == yr].mean()) for yr in (2013, 2014, 2015)}
oracle = {int(mm): prior_by_year[year_of_month[int(mm)]] for mm in np.unique(m_te)}
runs = {'Static (2012 threshold)': None,
        'Label-shift weighted, EM prior (label-free)':   ls_online(p_calib, y_cal_arr2, cal_m, p_te, y_te, m_te, alpha, 'em'),
        'Label-shift weighted, BBSE prior (label-free)': ls_online(p_calib, y_cal_arr2, cal_m, p_te, y_te, m_te, alpha, 'bbse'),
        'Label-shift weighted, oracle yearly prior':     ls_online(p_calib, y_cal_arr2, cal_m, p_te, y_te, m_te, alpha, oracle=oracle)}
rows = []
for name, r in runs.items():
    cov, size = (S_static[np.arange(T_te), y_te].astype(float), S_static.sum(1).astype(float)) if r is None else (r[0], r[1])
    for yr in (2013, 2014, 2015, 'All'):
        m_ = np.ones(T_te, bool) if yr == 'All' else yr_te == yr
        rows.append(dict(method=name, year=yr, cov=cov[m_].mean(), cov_default=cov[m_ & (y_te == 1)].mean(), cov_nondefault=cov[m_ & (y_te == 0)].mean(),
                         size=size[m_].mean(), mean_prior_used=(np.nan if r is None else r[2][m_].mean()),
                         realised_prior=(y_te[m_].mean())))
df_ls = pd.DataFrame(rows)
print(f"Calibration-year prior {y_cal_arr2.mean():.3f} (mean p_hat in 2012 = {p_calib.mean():.3f}).  alpha = 0.10, trailing window = 3 months of unlabelled scores.")
print(df_ls.round(4).to_string(index=False))

st18 = df_ls[(df_ls.method.str.startswith('Static')) & (df_ls.year == 2015)].iloc[0]
print("\nShare of the static 2015 coverage shortfall closed (0 % = nothing, 100 % = back to 90 %):")
for name in list(runs)[1:]:
    r18 = df_ls[(df_ls.method == name) & (df_ls.year == 2015)].iloc[0]
    print(f"  {name:48s} {100 * (r18['cov'] - st18['cov']) / max(1e-9, (1 - alpha) - st18['cov']):6.1f} %   (size {r18['size']:.3f} vs static {st18['size']:.3f})")
print("\nHow to read: if label-free prior-shift weighting closes most of the shortfall, prior shift is a sufficient explanation and a label-free adaptation exists;")
print("if it closes little while the oracle-prior row also fails, the drift is in P(Y|X) (concept drift), which no reweighting of class frequencies can fix.")

# monthly prior estimates against the realised monthly default rate
mths = np.unique(m_te); ph_em = runs['Label-shift weighted, EM prior (label-free)'][2]; ph_bb = runs['Label-shift weighted, BBSE prior (label-free)'][2]
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(mths, [y_te[m_te == mm].mean() for mm in mths], 'k.-', label='realised default rate (monthly)')
ax.plot(mths, [ph_em[m_te == mm][0] for mm in mths], 'r-', label='EM estimate (3-month trailing, unlabelled)')
ax.plot(mths, [ph_bb[m_te == mm][0] for mm in mths], 'b--', label='BBSE estimate')
ax.axhline(y_cal_arr2.mean(), color='grey', ls=':', label='2012 calibration prior'); ax.set_xlabel('month index'); ax.set_ylabel('P(default)')
ax.set_title('Can the class prior be tracked without labels?', fontweight='bold'); ax.legend(fontsize=8); plt.tight_layout()
save_fig('15c_prior_tracking'); plt.show()
df_ls.round(5).to_csv(f'{OUTDIR}/tables/15c_label_shift_conformal.csv', index=False)

## 16. Uncertainty Quantification & Sensitivity to Calibration Design
A single 2012 calibration draw has finite-sample noise, and 2012 is also one of the lowest-risk years. We therefore (a) **bootstrap** the 2012 calibration set, and (b) run a **rolling-origin sweep** over calibration year *and* window length (model retrained on all years before the calibration window; coverage measured 1, 2 and 3 years ahead).

In [ ]:
# (a) bootstrap the 2012 calibration set
rng = np.random.default_rng(123); B = 1000
def cov_from_q(q, mask=None):
    m = np.ones(len(y_te), bool) if mask is None else mask
    return evaluate(make_sets(p_te[m], q), y_te[m])
boot = []
for b in range(B):
    q = conf_q(rng.choice(cal_scores, len(cal_scores), replace=True), alpha)
    r = {'All': cov_from_q(q)['cov']}
    for yr in (2013, 2014, 2015): r[yr] = cov_from_q(q, yr_te == yr)['cov']
    r['default_cov'] = cov_from_q(q)['cov_d']
    boot.append(r)
boot = pd.DataFrame(boot)
print(f"Bootstrap of 2012 calibration set (B={B}), static split-conformal, alpha=0.10")
print(pd.DataFrame({'mean': boot.mean(), 'sd': boot.std(), 'p05': boot.quantile(.05), 'p95': boot.quantile(.95)}).round(4).to_string())

# (b) rolling origin: vary calibration year and window
def fit_model(tr):
    return lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1
                              ).fit(df_encoded.loc[tr, X_cols], y_all[tr])
rows = []
for c in (2010, 2011, 2012):
    for w in (1, 2, 3):
        tr = yr_all <= c - w; ca = (yr_all > c - w) & (yr_all <= c)
        if tr.sum() < 5000: continue          # too little history to train on
        mdl = fit_model(tr)
        pc = mdl.predict_proba(df_encoded.loc[ca, X_cols])[:, 1]
        q = conf_q(true_label_scores(pc, y_all[ca]), alpha)
        r = dict(cal_end=c, window_yrs=w, cal_default_rate=y_all[ca].mean(), n_cal=int(ca.sum()))
        for h in (1, 2, 3):
            mt = yr_all == c + h
            if mt.sum():
                pt = mdl.predict_proba(df_encoded.loc[mt, X_cols])[:, 1]
                r[f'cov_h{h}'] = evaluate(make_sets(pt, q), y_all[mt])['cov']
        rows.append(r)
ro = pd.DataFrame(rows); print("\nRolling-origin sweep (coverage h years after calibration ends):")
print(ro.round(4).to_string(index=False))
print("\nMean coverage by horizon:", ro[['cov_h1','cov_h2','cov_h3']].mean().round(4).to_dict())
vals = ro[['cov_h1','cov_h2','cov_h3']].values
print("Min coverage:", np.nanmin(vals).round(4), "| cells < 0.88:", int((vals < 0.88).sum()), "of", int(np.isfinite(vals).sum()))


## 16b. Combined uncertainty and a retrain-and-recalibrate baseline
**(a) Combined uncertainty.** §5's Wilson intervals treat the calibration threshold as fixed; §16's bootstrap resamples only the calibration set and treats the test year as fixed. The yearly coverage *gap* has both sources of noise, so we resample 2012 calibration scores **and** the test-year rows together and report the interval of (coverage − 0.90) and the share of draws that reach 0.90. (Rows are resampled independently; if loans within a month are correlated the true interval is somewhat wider.)

**(b) Retrain-and-recalibrate under label lag.** The adaptation baselines above keep the model frozen; the obvious practitioner alternative is to refit the model and recalibrate on more recent data. For a label-maturity lag of $L$ years, test year $Y$ can use only labels from years ≤ $Y-L$: the model is fit on years ≤ $Y-L-1$, the conformal threshold is calibrated on year $Y-L$, and coverage is measured on $Y$. Hyper-parameters are those of §3 (no tuning), so this isolates *recency of data*, not learner quality. $L=1$ with $Y=2013$ reproduces the base pipeline.

In [ ]:
# (a) combined calibration + test-row bootstrap of the yearly coverage gap
rng = np.random.default_rng(321); B = 2000
idx_yr = {yr: np.where(yr_te == yr)[0] for yr in (2013, 2014, 2015)}
draws = {yr: np.empty(B) for yr in idx_yr}
for b in range(B):
    q_b = conf_q(rng.choice(cal_scores, len(cal_scores), replace=True), alpha)
    hit = make_sets(p_te, q_b)[np.arange(len(y_te)), y_te]
    for yr, ii in idx_yr.items():
        draws[yr][b] = hit[rng.choice(ii, len(ii), replace=True)].mean()
rows = []
for yr, d_ in draws.items():
    m = yr_te == yr; hit0 = S_static[m][np.arange(m.sum()), y_te[m]]
    w_lo, w_hi = proportion_confint(int(hit0.sum()), int(m.sum()), alpha=0.05, method='wilson')
    rows.append(dict(Year=yr, coverage=hit0.mean(), wilson_test_only=f"[{w_lo:.4f}, {w_hi:.4f}]",
                     combined_95CI_gap_pp=f"[{100*(np.percentile(d_, 2.5)-0.9):+.2f}, {100*(np.percentile(d_, 97.5)-0.9):+.2f}]",
                     share_draws_ge_0p90=np.mean(d_ >= 0.90)))
print("Coverage gap vs 0.90 with calibration AND test-year resampling (B=2000):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

# (b) retrain + recalibrate under label lag L (years)
def retrain_recal(L, Y):
    cal_y = Y - L
    mdl = fit_model(yr_all <= cal_y - 1)
    ca, mt = yr_all == cal_y, yr_all == Y
    pc = mdl.predict_proba(df_encoded.loc[ca, X_cols])[:, 1]
    pt = mdl.predict_proba(df_encoded.loc[mt, X_cols])[:, 1]
    q = conf_q(true_label_scores(pc, y_all[ca]), alpha)
    e = evaluate(make_sets(pt, q), y_all[mt])
    return dict(L=L, Year=Y, train_end=cal_y - 1, cal_year=cal_y, q=q, cov=e['cov'], cov_default=e['cov_d'], size=e['size'],
                mean_pred=pt.mean(), actual=y_all[mt].mean(), AUC=roc_auc_score(y_all[mt], pt))
rt = pd.DataFrame([retrain_recal(L, Y) for L in (1, 2) for Y in (2013, 2014, 2015)])
fro = pd.DataFrame([dict(L='frozen', Year=yr, cov=evaluate(S_static[yr_te == yr], y_te[yr_te == yr])['cov'],
                         cov_default=evaluate(S_static[yr_te == yr], y_te[yr_te == yr])['cov_d'],
                         size=S_static[yr_te == yr].sum(1).mean(), mean_pred=p_te[yr_te == yr].mean(), actual=y_te[yr_te == yr].mean())
                    for yr in (2013, 2014, 2015)])
print("\nRetrain-and-recalibrate with labels known only up to Y-L, vs. the frozen 2007-11 model calibrated on 2012:")
print(pd.concat([rt, fro], ignore_index=True).round(4).to_string(index=False))
pd.concat([rt, fro], ignore_index=True).round(4).to_csv(f'{OUTDIR}/tables/16b_retrain_recalibrate.csv', index=False)
pd.DataFrame(rows).to_csv(f'{OUTDIR}/tables/16b_combined_bootstrap_gap.csv', index=False)

## 16c. Dependence-aware uncertainty, walk-forward γ, and retraining variance
(a) **Month-block bootstrap.** Loans issued in the same month share the macroeconomic environment; §16b resamples rows independently. Resampling whole months (calibration and test) gives the interval the paper should report; both are shown for comparison.
(b) **Walk-forward γ.** Sections 11–12 choose γ by looking at the years they evaluate. Here γ is chosen on an earlier test year whose labels have matured (validation year $=Y-1-\lfloor d/12\rfloor$) and applied to year $Y$, next to the hindsight-best γ; the difference is the optimism of the earlier tables.
(c) **Model-retraining variance.** §16 and §16b treat the fitted model as fixed. Twenty bagged refits (row and column subsampling, different seeds) show how much of the 2015 shortfall is noise in the base model.

In [ ]:
# ---------- §16c: month-block bootstrap, walk-forward selection of gamma, model-retraining variance ----------
cal_m = month_all[calib_mask.values]; y_cal_arr2 = y_calib.values
# (a) month-block vs row bootstrap of the yearly coverage gap (loans within a month share the macro environment)
rng = np.random.default_rng(2026); B = 2000
cal_blocks = [np.where(cal_m == mm)[0] for mm in np.unique(cal_m)]
te_blocks = {yr: [np.where(m_te == mm)[0] for mm in np.unique(m_te[yr_te == yr])] for yr in (2013, 2014, 2015)}
te_rows = {yr: np.where(yr_te == yr)[0] for yr in te_blocks}
d_blk = {yr: np.empty(B) for yr in te_blocks}; d_row = {yr: np.empty(B) for yr in te_blocks}
for b in range(B):
    pk = rng.integers(0, len(cal_blocks), len(cal_blocks)); q_blk = conf_q(cal_scores[np.concatenate([cal_blocks[i] for i in pk])], alpha)
    q_row = conf_q(rng.choice(cal_scores, len(cal_scores), replace=True), alpha)
    hit_blk = make_sets(p_te, q_blk)[np.arange(len(y_te)), y_te]; hit_row = make_sets(p_te, q_row)[np.arange(len(y_te)), y_te]
    for yr in te_blocks:
        bl = te_blocks[yr]; pk2 = rng.integers(0, len(bl), len(bl)); d_blk[yr][b] = hit_blk[np.concatenate([bl[i] for i in pk2])].mean()
        d_row[yr][b] = hit_row[rng.choice(te_rows[yr], len(te_rows[yr]), replace=True)].mean()
rows = []
for yr in te_blocks:
    for lab, dd in (('row bootstrap (iid)', d_row[yr]), ('month-block bootstrap', d_blk[yr])):
        lo_, hi_ = np.percentile(dd, [2.5, 97.5]); rows.append(dict(year=yr, resampling=lab, gap_lo_pp=100 * (lo_ - 0.9), gap_hi_pp=100 * (hi_ - 0.9),
                                                                   width_pp=100 * (hi_ - lo_), share_draws_ge_0p90=np.mean(dd >= 0.9)))
df_blk = pd.DataFrame(rows); print("Yearly coverage gap vs 0.90, calibration AND test resampled (B=2000):"); print(df_blk.round(3).to_string(index=False))
_w = df_blk.pivot(index='year', columns='resampling', values='width_pp')
print(f"-> month-block interval is wider than the iid row bootstrap in {int((_w['month-block bootstrap'] > _w['row bootstrap (iid)']).sum())} of {len(_w)} test years "
      f"(width ratio block/row: {', '.join(f'{k}: {v:.2f}' for k, v in (_w['month-block bootstrap'] / _w['row bootstrap (iid)']).items())}). Report the block version.\n")

# (b) walk-forward selection of gamma: choose gamma on an EARLIER test year only, apply to the later one
G0 = (0.0005, 0.001, 0.003, 0.005, 0.015, 0.05); wf = []
for d in (0, 12):
    G = G0 if d == 0 else tuple(G_DELAY_WF)           # delay > 0: batch-gain-scaled grid
    runs_g = {g: aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g, delay=d)[0] for g in G}
    for Y in (2014, 2015):
        Yv = Y - 1 - d // 12                                  # labels of the validation year must have matured by the start of year Y
        if Yv < 2013: continue
        cv = {g: runs_g[g][yr_te == Yv].mean() for g in G}; cy = {g: runs_g[g][yr_te == Y].mean() for g in G}
        g_sel = min(G, key=lambda g: abs(cv[g] - 0.9)); g_hs = min(G, key=lambda g: abs(cy[g] - 0.9))
        wf.append(dict(delay=d, test_year=Y, validation_year=Yv, gamma_selected=g_sel, cov_selected=cy[g_sel], gamma_hindsight=g_hs, cov_hindsight=cy[g_hs],
                       cov_gamma_0p015=cy.get(0.015, np.nan), cov_static=S_static[yr_te == Y][np.arange((yr_te == Y).sum()), y_te[yr_te == Y]].mean()))
df_wf = pd.DataFrame(wf); print("Walk-forward gamma selection (validation year strictly earlier than the test year) vs hindsight-best gamma:")
print(df_wf.round(4).to_string(index=False))
print("-> the gap between cov_selected and cov_hindsight is the optimism of tuning gamma on the evaluation year, as done in Sections 11-12.\n")

# (c) model-retraining variance (bagged LightGBM, 20 seeds)
N_SEEDS = 20; rv = []
for sd in range(N_SEEDS):
    mdl = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                             random_state=sd, verbose=-1).fit(X_train, y_train)
    pc, pt = mdl.predict_proba(X_calib)[:, 1], mdl.predict_proba(X_test)[:, 1]
    S_ = make_sets(pt, conf_q(true_label_scores(pc, y_cal_arr2), alpha)); r = dict(seed=sd, AUC_2015=roc_auc_score(y_te[yr_te == 2015], pt[yr_te == 2015]))
    for yr in (2013, 2014, 2015): r[f'cov_{yr}'] = evaluate(S_[yr_te == yr], y_te[yr_te == yr])['cov']
    rv.append(r)
df_rv = pd.DataFrame(rv)
print(f"Static split-conformal coverage across {N_SEEDS} retrainings of the base model (subsample 0.8, colsample 0.8), mean +- sd [min, max]:")
for c in ('AUC_2015', 'cov_2013', 'cov_2014', 'cov_2015'): print(f"  {c:9s} {df_rv[c].mean():.4f} +- {df_rv[c].std():.4f}  [{df_rv[c].min():.4f}, {df_rv[c].max():.4f}]")
df_blk.round(4).to_csv(f'{OUTDIR}/tables/16c_block_bootstrap.csv', index=False); df_wf.round(4).to_csv(f'{OUTDIR}/tables/16c_walk_forward_gamma.csv', index=False)
df_rv.round(5).to_csv(f'{OUTDIR}/tables/16c_retraining_variance.csv', index=False)

## 17. Sweep over the Miscoverage Level $\alpha$

In [ ]:
rows = []
for a in (0.05, 0.10, 0.20):
    ca_ = cal_scores; qs = conf_q(ca_, a); Ss = make_sets(p_te, qs)
    q0_, q1_ = mondrian_q(p_calib, y_calib.values, a); Sm = make_sets(p_te, q0_, q1_)
    c0, s0, _ = aci(p_te, y_te, m_te, ca_, a, 0.015, delay=0)
    c12, s12, _ = aci(p_te, y_te, m_te, ca_, a, 0.001, delay=12)
    es, em = evaluate(Ss, y_te), evaluate(Sm, y_te)
    rows.append(dict(alpha=a, target=1-a,
        static_cov=es['cov'], static_cov_2015=evaluate(Ss[yr_te==2015], y_te[yr_te==2015])['cov'],
        static_cov_default=es['cov_d'], static_size=es['size'],
        mond_cov_default=em['cov_d'], mond_size=em['size'],
        ACI_cov=c0.mean(), ACI_size=s0.mean(), ACIdelay12_cov=c12.mean(), ACIdelay12_cov_2015=c12[yr_te==2015].mean(), ACIdelay12_size=s12.mean()))
print(pd.DataFrame(rows).round(4).T.to_string(header=False))


## 18. Baseline Comparison: Logistic Regression & Post-hoc Calibration
Two questions: (a) does a simpler logistic-regression baseline match or beat LightGBM out-of-time; (b) do Platt scaling or isotonic regression change the conformal coverage picture. In (b) the recalibrator is fit on a half of the 2012 calibration set disjoint from the half that sets the conformal threshold, which preserves exchangeability. Section 18c then reruns the conformal pipeline on the logistic base model.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

# ---- (a) Logistic regression baseline ----
scaler = StandardScaler().fit(X_train)
logreg = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(scaler.transform(X_train), y_train)

perf_lookup = df_perf.set_index('Year')
lr_rows = []
for yr in range(2012, 2016):
    m = yr_all == yr
    p_lr = logreg.predict_proba(scaler.transform(df_encoded.loc[m, X_cols]))[:, 1]
    lr_rows.append(dict(Year=yr,
        AUC_LogReg=roc_auc_score(y_all[m], p_lr), AUC_LightGBM=perf_lookup.loc[yr, 'ROC-AUC'],
        Brier_LogReg=brier_score_loss(y_all[m], p_lr), Brier_LightGBM=perf_lookup.loc[yr, 'Brier Score']))
df_lr = pd.DataFrame(lr_rows)
df_lr['LogReg_wins_AUC'] = df_lr.AUC_LogReg > df_lr.AUC_LightGBM
print("Logistic regression vs. LightGBM, yearly AUC / Brier:")
print(df_lr.round(4).to_string(index=False))
win_years = df_lr.loc[df_lr.LogReg_wins_AUC, 'Year'].tolist()
print(f"\nLogistic regression has HIGHER AUC than the (untuned, fixed-hyperparameter) LightGBM in: {win_years if win_years else 'no years'}")
df_lr.round(4).to_csv(f'{OUTDIR}/tables/18a_logreg_vs_lightgbm.csv', index=False)

In [ ]:
# ---- (b) Post-hoc calibration of the LightGBM probabilities
# The 2012 set is split at random: half A fits the recalibrator, half B sets the conformal quantile, so the scores that set the threshold are out-of-sample
# for the recalibrator (exchangeability is preserved). The raw model is evaluated with the same half-B threshold so the three rows are like-for-like.
eps = 1e-6
def to_logit(p):
    return np.log(np.clip(p, eps, 1 - eps) / np.clip(1 - p, eps, 1 - eps)).reshape(-1, 1)

y_cal = y_calib.values
perm = np.random.default_rng(SEED).permutation(len(p_calib))
iA, iB = perm[:len(perm) // 2], perm[len(perm) // 2:]
platt = LogisticRegression().fit(to_logit(p_calib[iA]), y_cal[iA])
iso = IsotonicRegression(out_of_bounds='clip').fit(p_calib[iA], y_cal[iA])

transforms = {
    'Raw LightGBM': lambda p: p,
    'Platt-scaled':  lambda p: platt.predict_proba(to_logit(p))[:, 1],
    'Isotonic':      lambda p: iso.predict(p),
}

calib_rows = []
for name, tr in transforms.items():
    pB = tr(p_calib[iB])
    scores_t = 1 - np.where(y_cal[iB] == 1, pB, 1 - pB)
    q_t = conf_q(scores_t, alpha)
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
        e = evaluate(make_sets(tr(p_te[m]), q_t), y_te[m])
        calib_rows.append(dict(Calibration=name, Year=yr, Coverage=e['cov'], CovDefault=e['cov_d'], SetSize=e['size']))
df_calib_cmp = pd.DataFrame(calib_rows)
print(f"Effect of post-hoc calibration on split-conformal coverage (recalibrator fit on {len(iA)} rows of 2012, threshold on the other {len(iB)}):")
print(df_calib_cmp.pivot_table(index='Calibration', columns='Year', values='Coverage').round(4).to_string())
print("\nBrier score of each map on the 2015 test year (lower is better):")
m18 = yr_te == 2015
for name, tr in transforms.items():
    print(f"  {name:13s} {brier_score_loss(y_te[m18], tr(p_te[m18])):.4f}")

print("\nTakeaway: conformal *validity* needs exchangeability, not a calibrated probability, so Platt/isotonic maps fit on 2012")
print("cannot repair drift in P(Y|X) that appears afterwards; they only correct miscalibration inside 2012. Differences between")
print("rows are of the order of the calibration-half sampling noise (n_B is half of 2012) -- see the Section 16 bootstrap for scale.")
df_calib_cmp.round(4).to_csv(f'{OUTDIR}/tables/18b_posthoc_calibration.csv', index=False)

## 18c. Conformal Prediction on the Logistic-Regression Base Model
Section 18a shows logistic regression out-ranking LightGBM in every year, yet every conformal result above uses LightGBM. If the coverage failure or the Mondrian/ACI behaviour were an artefact of the weaker learner, the conclusions would not transfer. Here the identical static / Mondrian / ACI pipeline (same 2012 calibration set, same $\alpha$, same $\gamma$) is run on the logistic model.

In [ ]:
p_lr_all = logreg.predict_proba(scaler.transform(df_encoded[X_cols]))[:, 1]
p_lr_cal, p_lr_te = p_lr_all[calib_mask.values], p_lr_all[te]
cal_sc_lr = true_label_scores(p_lr_cal, y_calib.values)

S_lr = make_sets(p_lr_te, conf_q(cal_sc_lr, alpha))
q0_lr, q1_lr = mondrian_q(p_lr_cal, y_calib.values, alpha)
S_lr_m = make_sets(p_lr_te, q0_lr, q1_lr)
c_lr, s_lr, _ = aci(p_lr_te, y_te, m_te, cal_sc_lr, alpha, gamma=0.015)

rows = []
for base, sets_static, sets_mond, cov_aci, size_aci in (
        ('LightGBM',     S_static, S_mond, cov_a, size_a),
        ('LogReg',       S_lr,     S_lr_m, c_lr,  s_lr)):
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
        es, em = evaluate(sets_static[m], y_te[m]), evaluate(sets_mond[m], y_te[m])
        rows.append(dict(Base=base, Year=yr,
                         Static_cov=es['cov'], Static_cov_default=es['cov_d'], Static_size=es['size'],
                         Mond_cov=em['cov'], Mond_cov_default=em['cov_d'], Mond_cov_nondefault=em['cov_nd'], Mond_size=em['size'],
                         ACI_cov=cov_aci[m].mean(), ACI_size=size_aci[m].mean()))
df_lr_conf = pd.DataFrame(rows)
print("Same conformal pipeline on both base models (alpha=0.10, gamma=0.015):")
print(df_lr_conf.round(4).to_string(index=False))

m18 = yr_te == 2015
print(f"\n2015 mean predicted risk vs realised default rate -- LightGBM: {p_te[m18].mean():.3f} vs {y_te[m18].mean():.3f}"
      f" | LogReg: {p_lr_te[m18].mean():.3f} vs {y_te[m18].mean():.3f}")
df_lr_conf.round(4).to_csv(f'{OUTDIR}/tables/18c_conformal_on_logreg.csv', index=False)

## 18d. Validation-tuned LightGBM as a third base model
§18a showed the fixed 150-tree LightGBM losing to logistic regression in every year, and §18c added logistic regression but left the strongest tree learner untried. Here depth and number of trees are chosen by log-loss on **2011** (model fit on ≤ 2010, no test-year or 2012 data touched), then refit on ≤ 2011 with the selected size. The identical static / Mondrian / ACI pipeline is applied with the same 2012 calibration set.

In [ ]:
tr13, va14 = yr_all <= 2010, yr_all == 2011
Xtr13, Xva14 = df_encoded.loc[tr13, X_cols], df_encoded.loc[va14, X_cols]
best = dict(ll=np.inf); g_rows = []
for depth in (2, 3, 5):
    m_ = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, max_depth=depth, min_child_samples=50,
                            random_state=SEED, verbose=-1).fit(Xtr13, y_all[tr13])
    for k in range(25, 601, 25):
        ll = log_loss(y_all[va14], m_.predict_proba(Xva14, num_iteration=k)[:, 1])
        g_rows.append(dict(depth=depth, n_trees=k, val_logloss_2011=ll))
        if ll < best['ll']: best = dict(ll=ll, depth=depth, k=k)
print(f"Selected on 2011 validation: max_depth={best['depth']}, n_estimators={best['k']} (val log-loss {best['ll']:.4f})")
lgb_tuned = lgb.LGBMClassifier(n_estimators=best['k'], learning_rate=0.05, max_depth=best['depth'], min_child_samples=50,
                               random_state=SEED, verbose=-1).fit(X_train, y_train)
p_tu_all = lgb_tuned.predict_proba(df_encoded[X_cols])[:, 1]
p_tu_cal, p_tu_te = p_tu_all[calib_mask.values], p_tu_all[te]
cal_sc_tu = true_label_scores(p_tu_cal, y_calib.values)

S_tu = make_sets(p_tu_te, conf_q(cal_sc_tu, alpha))
q0_tu, q1_tu = mondrian_q(p_tu_cal, y_calib.values, alpha)
S_tu_m = make_sets(p_tu_te, q0_tu, q1_tu)
c_tu, s_tu, _ = aci(p_tu_te, y_te, m_te, cal_sc_tu, alpha, gamma=0.015)

rows = []
for yr in (2013, 2014, 2015, 'All'):
    m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
    es, em = evaluate(S_tu[m], y_te[m]), evaluate(S_tu_m[m], y_te[m])
    rows.append(dict(Base='LightGBM (tuned)', Year=yr, Static_cov=es['cov'], Static_cov_default=es['cov_d'], Static_size=es['size'],
                     Mond_cov=em['cov'], Mond_cov_default=em['cov_d'], Mond_cov_nondefault=em['cov_nd'], Mond_size=em['size'],
                     ACI_cov=c_tu[m].mean(), ACI_size=s_tu[m].mean()))
df_tu = pd.DataFrame(rows)
print("Tuned LightGBM, same conformal pipeline (alpha=0.10, gamma=0.015):")
print(df_tu.round(4).to_string(index=False))
auc_rows = [dict(Year=yr, AUC_tuned=roc_auc_score(y_all[yr_all == yr], p_tu_all[yr_all == yr]),
                 AUC_fixedLGB=perf_lookup.loc[yr, 'ROC-AUC'], AUC_LogReg=df_lr.set_index('Year').loc[yr, 'AUC_LogReg'],
                 Brier_tuned=brier_score_loss(y_all[yr_all == yr], p_tu_all[yr_all == yr])) for yr in range(2012, 2016)]
print("\nDiscrimination by year:"); print(pd.DataFrame(auc_rows).round(4).to_string(index=False))
pd.concat([df_lr_conf, df_tu]).round(4).to_csv(f'{OUTDIR}/tables/18d_conformal_three_base_models.csv', index=False)
pd.DataFrame(g_rows).round(5).to_csv(f'{OUTDIR}/tables/18d_lgbm_validation_grid.csv', index=False)

## 18e. Ablation: without the lender-assigned `int_rate`
§2b warned that `int_rate` is set by the lender's own risk model, so a credit-risk model that uses it may partly re-state the lender's earlier score, and asked for results with and without it; that was never run. The identical pipeline is refitted without the column.

In [ ]:
# ---------- §18e int_rate ablation: does the coverage / drift story survive without the lender-assigned rate? ----------
X_cols_ni = [c for c in X_cols if c != 'int_rate']
lgb_ni = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1).fit(df_encoded.loc[train_mask, X_cols_ni], y_train)
p_ni_all = lgb_ni.predict_proba(df_encoded[X_cols_ni])[:, 1]
p_ni_cal, p_ni_te = p_ni_all[calib_mask.values], p_ni_all[te]
cs_ni = true_label_scores(p_ni_cal, y_calib.values)
S_ni = make_sets(p_ni_te, conf_q(cs_ni, alpha)); q0_ni, q1_ni = mondrian_q(p_ni_cal, y_calib.values, alpha); S_ni_m = make_sets(p_ni_te, q0_ni, q1_ni)
c_ni, s_ni, _ = aci(p_ni_te, y_te, m_te, cs_ni, alpha, gamma=0.015)

rows = []
for base, p_all_b, S_s, S_m, c_ac, s_ac in (('LightGBM, all features', p_all, S_static, S_mond, cov_a, size_a),
                                            ('LightGBM, no int_rate', p_ni_all, S_ni, S_ni_m, c_ni, s_ni)):
    for yr in (2012, 2013, 2014, 2015):
        mk_ = yr_all == yr
        r = dict(base=base, year=yr, AUC=roc_auc_score(y_all[mk_], p_all_b[mk_]), Brier=brier_score_loss(y_all[mk_], p_all_b[mk_]),
                 mean_pred=p_all_b[mk_].mean(), actual=y_all[mk_].mean())
        if yr >= 2013:
            m_ = yr_te == yr; es, em = evaluate(S_s[m_], y_te[m_]), evaluate(S_m[m_], y_te[m_])
            r.update(static_cov=es['cov'], static_cov_default=es['cov_d'], static_size=es['size'], mond_cov_default=em['cov_d'],
                     mond_cov_nondefault=em['cov_nd'], ACI_cov=c_ac[m_].mean(), ACI_size=s_ac[m_].mean())
        rows.append(r)
df_ni = pd.DataFrame(rows); print("Ablation: same pipeline with and without int_rate (alpha=0.10, gamma=0.015):"); print(df_ni.round(4).to_string(index=False))
print("\nIf the AUC drops sharply without int_rate, the 'credit-risk model' was partly re-stating the lender's own score; if the calibration-gap growth and the 2015")
print("coverage shortfall persist, the shift findings do not hinge on that variable.")
df_ni.round(4).to_csv(f'{OUTDIR}/tables/18e_int_rate_ablation.csv', index=False)

## 19. DtACI and Non-Exchangeable Weighted Conformal
Two shift-aware methods:
1. **DtACI** (Gibbs & Candes, 2022): instead of committing to one gamma, run several ACI "experts" with different gammas in parallel and combine them with online exponential weighting. *Simplification, stated explicitly*: this implementation aggregates the experts' alpha_t by a deterministic weighted average rather than the paper's stochastic single-expert sampling scheme, and updates every expert's own alpha_t^k from its own realised error -- it is a practical approximation to DtACI, not a byte-for-byte reproduction.
2. **Non-exchangeable weighted conformal** (Barber et al., 2023): weight calibration points by a *fixed* function of recency (time distance to the test point) instead of a covariate-dependent density ratio (contrast with the covariate-shift weights of §15). Applied here within the single 2012 calibration window, so it can only re-weight *which months of 2012* matter more for a given test point -- it cannot borrow information from 2013-2015, which bounds how much it can help.

**Implementation notes.** DtACI's expert-weight update uses the pinball loss of quantile level $1-\alpha$ in score space, $\rho(s-q)=(1-\alpha)(s-q)$ if $s>q$ else $\alpha(q-s)$, evaluated at the *target* $\alpha$; $\eta$ is swept because at $\eta=0.02$ the weights can barely move (§19c selects it walk-forward). The recency-weighted quantile includes the test point's own unit weight (Barber et al.'s $+\delta_\infty$ mass), so $\rho=1$ reproduces standard split conformal exactly (asserted). Consequence: the threshold is $+\infty$ (full set) whenever the effective calibration weight is below $(1-\alpha)/\alpha = 9$, which happens at aggressive decay for test months far from 2012, so the cell reports set size and the share of infinite thresholds next to coverage.


In [ ]:
# ---- (a) DtACI: multiple ACI experts (different gamma) combined by exponential weighting ----
def dtaci(p, y, cal_scores, target_alpha=0.10, gammas=(0.001, 0.005, 0.02, 0.08), eta=0.02, sigma=0.002, clip=(0.01, 0.99)):
    K = len(gammas); g = np.asarray(gammas, float)
    s_sorted = np.sort(cal_scores); n = len(s_sorted)

    def q_of(alpha_):
        a_c = np.clip(alpha_, *clip)
        k = int(np.ceil((n + 1) * (1 - a_c)))
        return s_sorted[min(k, n) - 1]

    a = np.full(K, target_alpha)          # each expert's own alpha_t
    w = np.ones(K) / K                    # expert weights
    T = len(y)
    alpha_hist = np.zeros(T); cov = np.zeros(T); size = np.zeros(T); w_hist = np.zeros((T, K))

    for t in range(T):
        q_experts = np.array([q_of(a[k]) for k in range(K)])
        alpha_t = float(np.dot(w, a))                       # combined working alpha
        q_t = q_of(alpha_t)
        p_t, y_t = p[t], y[t]

        set0_t, set1_t = p_t <= q_t, (1 - p_t) <= q_t
        covered_t = set1_t if y_t == 1 else set0_t

        set0_k, set1_k = p_t <= q_experts, (1 - p_t) <= q_experts   # each expert's own set
        err_k = 1.0 - (set1_k if y_t == 1 else set0_k).astype(float)

        # pinball loss of quantile level (1 - target_alpha) in score space, using the TARGET alpha
        s_t = (1 - p_t) if y_t == 1 else p_t
        pin = np.where(s_t > q_experts, (1 - target_alpha) * (s_t - q_experts), target_alpha * (q_experts - s_t))
        w = w * np.exp(-eta * pin)
        w = (1 - sigma) * (w / w.sum()) + sigma / K          # sleeping-expert mixing keeps all alive

        a = np.clip(a + g * (target_alpha - err_k), *clip)

        alpha_hist[t] = alpha_t; cov[t] = covered_t; size[t] = int(set0_t) + int(set1_t); w_hist[t] = w
    return cov, size, alpha_hist, w_hist

gammas = (0.001, 0.005, 0.02, 0.08)
rows, final_w = [], {}
for eta in (0.02, 0.2, 1.0):
    cov_dt, size_dt, alpha_dt, w_dt = dtaci(p_te, y_te, cal_scores, gammas=gammas, eta=eta)
    final_w[eta] = w_dt[-1]
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
        rows.append(dict(eta=eta, Year=yr, DtACI_cov=cov_dt[m].mean(), DtACI_size=size_dt[m].mean(),
                         Static_cov=evaluate(S_static[m], y_te[m])['cov'], ACI_g015_cov=cov_a[m].mean(),
                         alpha_t_mean=alpha_dt[m].mean(), alpha_t_std=alpha_dt[m].std()))
df_dtaci = pd.DataFrame(rows)
print("DtACI (corrected loss) vs. static vs. single-gamma ACI (gamma=0.015), by learning rate eta:")
print(df_dtaci.round(4).to_string(index=False))
print("\nFinal expert weights by eta (uniform = 0.25 each -> the weighting is doing nothing):")
for eta, w_ in final_w.items():
    print(f"  eta={eta:<5} " + "  ".join(f"g={g_}: {x:.3f}" for g_, x in zip(gammas, w_)) + f"   max|w-0.25| = {np.abs(w_ - 0.25).max():.3f}")
df_dtaci.round(4).to_csv(f'{OUTDIR}/tables/19a_dtaci.csv', index=False)

In [ ]:
# ---- (b) Barber et al. (2023)-style non-exchangeable, recency-weighted conformal ----
def barber_weighted_q(cal_scores, cal_months, test_months, rho, alpha_level):
    """Weight calibration point i by rho**|month_i - month_test|, independent of covariates.
    The test point's own weight (rho**0 = 1, the +infinity mass of the weighted quantile) is part of the normaliser, so rho=1 reproduces
    standard split conformal exactly (asserted below)."""
    order = np.argsort(cal_scores)
    s_sorted, months_sorted = cal_scores[order], cal_months[order]
    test_months = np.asarray(test_months)
    q = np.empty(len(test_months))
    for tm in np.unique(test_months):
        w = rho ** np.abs(months_sorted - tm)
        cum = np.cumsum(w)
        idx = np.searchsorted(cum, (1 - alpha_level) * (w.sum() + 1.0))
        q[test_months == tm] = np.inf if idx >= len(s_sorted) else s_sorted[idx]
    return q

cal_months = month_all[calib_mask.values]
barber_rows = []
for rho in (1.00, 0.98, 0.90, 0.75):
    for yr in (2013, 2014, 2015):
        m = yr_te == yr
        q = barber_weighted_q(cal_scores, cal_months, m_te[m], rho, alpha)
        p_m, y_m = p_te[m], y_te[m]
        in0, in1 = p_m <= q, (1 - p_m) <= q
        covered = np.where(y_m == 1, in1, in0)
        barber_rows.append(dict(rho=rho, Year=yr, coverage=covered.mean(),
                                 size=(in0.astype(int) + in1.astype(int)).mean(),
                                 share_inf_threshold=np.mean(~np.isfinite(q))))
df_barber = pd.DataFrame(barber_rows)

# identity check: rho = 1 must equal static split conformal
for yr in (2013, 2014, 2015):
    m = yr_te == yr
    cov_rho1 = df_barber[(df_barber.rho == 1.0) & (df_barber.Year == yr)]['coverage'].iloc[0]
    assert np.isclose(cov_rho1, evaluate(S_static[m], y_te[m])['cov']), f"rho=1 != static in {yr}"

print("Barber et al. (2023)-style recency-weighted conformal, by decay rate rho (rho=1.0 = unweighted static, verified equal):")
print(df_barber.pivot_table(index='rho', columns='Year', values='coverage').round(4).to_string())
print("\nMean set size:")
print(df_barber.pivot_table(index='rho', columns='Year', values='size').round(4).to_string())
print("\nShare of test points whose threshold is +infinity (full set {0,1}, i.e. the method abstains):")
print(df_barber.pivot_table(index='rho', columns='Year', values='share_inf_threshold').round(4).to_string())
print("\nTakeaway: the test point carries its own unit weight (the +infinity mass), so once the effective calibration weight for a test month")
print("falls below (1-alpha)/alpha = 9 the threshold is infinite. At aggressive decay a distant test month therefore gets full sets and")
print("coverage approaches 1 -- coverage bought with trivial sets, not an improvement. Read coverage together with size and the")
print("infinite-threshold share. Because the calibration window is a single year (2012), the method can only re-rank which 2012 months")
print("matter for a test point and cannot see 2013-2015; it needs a calibration window that spans the drift to be useful.")
df_barber.round(4).to_csv(f'{OUTDIR}/tables/19b_barber_weighted.csv', index=False)

## 19c. Walk-forward selection of step sizes and decay rates
$\gamma$ (ACI), $\eta$ (DtACI, quantile tracker) and $\rho$ (recency weights) are compared above on the same years they are evaluated on, which flatters every adaptive method. Here each is chosen on the **previous** year — the setting whose coverage there is closest to $1-\alpha$, ties broken by smaller mean set size — and applied to the next year with the same 2012 calibration set. The hindsight-best setting and the worst setting in the grid are shown beside it: the first measures the optimism of the earlier tables, the second how much the method depends on its tuning parameter at all. Immediate feedback only (DtACI and the recency weights have no delayed-label implementation here), and only 2014 and 2015 can be validated on a preceding year.

In [ ]:
# ---------- §19c nested (walk-forward) selection of the step sizes / decay rates ----------
# Selection rule: on the validation year, pick the setting whose coverage is closest to 1-alpha (ties -> smaller mean set size).
# Validation year = Y-1 (labels matured because delay=0 here). The selected setting is then applied to year Y with the SAME 2012 calibration set.
# Methods without a delay implementation (DtACI, recency-weighted) are evaluated with immediate feedback only.
hit_static = S_static[np.arange(T_te), y_te].astype(float)
grids = {'ACI gamma':           {g: aci(p_te, y_te, m_te, cal_scores, alpha, gamma=g)[:2] for g in (0.0005, 0.001, 0.005, 0.015, 0.05)},
         'Quantile tracker eta': {e: quantile_tracker(p_te, y_te, m_te, q_static, alpha, e)[:2] for e in (0.001, 0.002, 0.005, 0.02)},
         'DtACI eta':           {e: dtaci(p_te, y_te, cal_scores, gammas=(0.001, 0.005, 0.02, 0.08), eta=e)[:2] for e in (0.02, 0.2, 1.0)}}
rec_runs = {}
for rho in (1.00, 0.98, 0.90, 0.75):
    q_ = barber_weighted_q(cal_scores, cal_months, m_te, rho, alpha); i0, i1 = p_te <= q_, (1 - p_te) <= q_
    rec_runs[rho] = (np.where(y_te == 1, i1, i0).astype(float), (i0.astype(int) + i1.astype(int)).astype(float))
grids['Recency-weighted rho'] = rec_runs

rows = []
for meth, runs_ in grids.items():
    for Y in (2014, 2015):
        Yv = Y - 1; mv, mY = yr_te == Yv, yr_te == Y
        key = lambda k: (abs(runs_[k][0][mv].mean() - (1 - alpha)), runs_[k][1][mv].mean())
        k_sel = min(runs_, key=key); k_hs = min(runs_, key=lambda k: (abs(runs_[k][0][mY].mean() - (1 - alpha)), runs_[k][1][mY].mean()))
        worst = max(runs_, key=lambda k: abs(runs_[k][0][mY].mean() - (1 - alpha)))
        rows.append(dict(method=meth, test_year=Y, validation_year=Yv, param_selected=k_sel, cov_selected=runs_[k_sel][0][mY].mean(), size_selected=runs_[k_sel][1][mY].mean(),
                         param_hindsight=k_hs, cov_hindsight=runs_[k_hs][0][mY].mean(), cov_worst_in_grid=runs_[worst][0][mY].mean(),
                         cov_static=hit_static[mY].mean(), size_static=S_static[mY].sum(1).mean()))
df_nested = pd.DataFrame(rows)
df_nested['optimism_pp'] = 100 * (abs(df_nested.cov_selected - (1 - alpha)) - abs(df_nested.cov_hindsight - (1 - alpha)))
print("Walk-forward selection (validation = previous year, immediate feedback) vs hindsight-best setting vs worst setting in the grid:")
print(df_nested.round(4).to_string(index=False))
print("\nRead: cov_selected is the honest number to report for each method. optimism_pp = how much closer to 0.90 the hindsight setting looks than the selected one (0 = selection was already optimal).")
print("cov_worst_in_grid shows how much the method's result depends on the tuning parameter at all. Only two test years (2014, 2015) can be validated on the preceding year, so treat the table as indicative.")
df_nested.round(5).to_csv(f'{OUTDIR}/tables/19c_nested_selection.csv', index=False)

## 20. Revised Decision-Utility Analysis
Section 8's first-cut table pools 2013–2015, has no cost model or baseline, and uses marginal sets that Section 13 showed under-cover defaults. This section redoes it per year.

**What the conformal sets are, for a binary label.** With score $1-\hat p(y\mid x)$, class 0 is in the set iff $\hat p \le \hat q$ and class 1 iff $\hat p \ge 1-\hat q$. So *static conformal is exactly a symmetric probability cutoff*: approve if $\hat p < 1-\hat q$, reject if $\hat p > \hat q$, review in between. Mondrian conformal is an asymmetric two-threshold cutoff (approve if $\hat p<1-\hat q_1$, reject if $\hat p>\hat q_0$). Conformal prediction only *chooses the thresholds*, by targeting coverage. The honest baseline question is therefore whether that way of choosing thresholds beats choosing them directly for the objective, so the comparison is against **cost-optimal cutoffs fit on the 2012 calibration set**, plus trivial policies.

1. **Per-year default rate and share of loans** by tier for Static, Mondrian and the cost-optimal cutoffs.
2. **An explicitly illustrative cost model** (not real underwriting economics): approving a defaulter costs a fraction of the loan (loss given default); rejecting a good loan costs a smaller fraction (foregone margin); a manual review costs a flat fee **and** the reviewer is imperfect (a share of reviewed defaulters is still approved, a share of reviewed good loans is still rejected).
3. **Trivial policies** (approve all, reject all, review all) and a sensitivity grid over review cost and reviewer accuracy, with the cost-optimal cutoffs re-fit for each cell.

**Design choices.** The reviewer is imperfect by construction: with a perfect reviewer at a flat fee, reviewing everything would beat every policy and the comparison would be vacuous. A "plain probability cutoff" is not a separate baseline, because static conformal already *is* one (the identity is asserted below); the baselines are therefore cost-optimal cutoffs and trivial policies.


In [ ]:
def action_tiers(sets):
    """Map each boolean (n,2) membership row to 'approve' / 'reject' / 'review'."""
    tiers = np.full(sets.shape[0], 'review', dtype=object)
    tiers[sets[:, 0] & ~sets[:, 1]] = 'approve'
    tiers[sets[:, 1] & ~sets[:, 0]] = 'reject'
    return tiers

def tier_rows(tiers, y, yr_arr, label):
    rows = []
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y), bool) if yr == 'All' else yr_arr == yr
        for tname in ('approve', 'review', 'reject'):
            tm = m & (tiers == tname)
            rows.append(dict(Method=label, Year=yr, Tier=tname, N=int(tm.sum()),
                              Pct_of_year=tm.sum() / m.sum() if m.sum() else np.nan,
                              DefaultRate=y[tm].mean() if tm.sum() else np.nan))
    return rows

# (a) Static and Mondrian tiers, from the sets already built in Sections 5 and 13
static_tiers = action_tiers(S_static)
mond_tiers = action_tiers(S_mond)

# Identity: static split conformal on a binary label IS a symmetric probability cutoff [1-q, q] (needs q > 0.5).
assert q_static > 0.5, ("q_static <= 0.5: with a base default rate this low the 90 % conformal sets can be EMPTY for some loans and the "
                        "symmetric-cutoff identity used in Sections 20/20b does not hold; check the data (real 36-month data has ~12-17 % defaults).")
sym_tiers_arr = np.where(p_te < 1 - q_static, 'approve', np.where(p_te > q_static, 'reject', 'review'))
assert np.array_equal(static_tiers, sym_tiers_arr), "static conformal tiers != symmetric cutoff [1-q, q]"
print(f"Static conformal == symmetric cutoff: approve if p < {1 - q_static:.3f}, reject if p > {q_static:.3f} (verified on every test row).")
print(f"Mondrian conformal == asymmetric cutoff: approve if p < {1 - q1:.3f}, reject if p > {q0:.3f}.\n")

# ---- (c) simple, explicitly illustrative cost model (expected-value form) ----
LOSS_GIVEN_DEFAULT = 0.60   # fraction of loan_amnt lost when an approved loan defaults
FOREGONE_MARGIN   = 0.08    # fraction of loan_amnt in margin foregone by wrongly rejecting a good loan
REVIEW_COST       = 75.0    # flat handling cost per manually reviewed case
REVIEWER_MISS     = 0.30    # share of reviewed DEFAULTERS the reviewer still approves (pays loss-given-default)
REVIEWER_FALSE_REJECT = 0.10  # share of reviewed GOOD loans the reviewer still rejects (pays foregone margin)

loan_te = df_encoded.loc[test_mask, 'loan_amnt'].values
loan_cal = df_encoded.loc[calib_mask, 'loan_amnt'].values
y_cal_arr = y_calib.values

def case_costs(y, loan, review_cost, miss, false_reject):
    """Per-loan cost under each action: (approve, reject, review)."""
    ca = LOSS_GIVEN_DEFAULT * loan * y
    cr = FOREGONE_MARGIN * loan * (1 - y)
    cv = review_cost + miss * LOSS_GIVEN_DEFAULT * loan * y + false_reject * FOREGONE_MARGIN * loan * (1 - y)
    return ca, cr, cv

def strategy_cost(tiers, y, loan, review_cost=REVIEW_COST, miss=REVIEWER_MISS, false_reject=REVIEWER_FALSE_REJECT):
    ca, cr, cv = case_costs(y, loan, review_cost, miss, false_reject)
    return np.where(tiers == 'approve', ca, np.where(tiers == 'reject', cr, cv))

# (b) Baseline: cost-optimal cutoff pair (approve if p <= lo, reject if p >= hi), fit on the 2012 calibration set.
def cost_optimal_cutoffs(p, y, loan, review_cost, miss, false_reject, grid=121):
    o = np.argsort(p); ps = p[o]
    ca, cr, cv = case_costs(y[o], loan[o], review_cost, miss, false_reject)
    A, R, V = (np.r_[0.0, np.cumsum(c)] for c in (ca, cr, cv))
    n = len(p); idx = np.unique(np.linspace(0, n, grid).astype(int))
    best_cost, best = np.inf, (0, n)
    for i in idx:                      # the i lowest-p loans are approved ...
        for j in idx[idx >= i]:        # ... loans from position j upward are rejected, the rest reviewed
            c = A[i] + (V[j] - V[i]) + (R[n] - R[j])
            if c < best_cost: best_cost, best = c, (i, j)
    i, j = best
    return (ps[i - 1] if i > 0 else -np.inf), (ps[j] if j < n else np.inf)

def apply_cutoffs(p, lo, hi):
    return np.where(p <= lo, 'approve', np.where(p >= hi, 'reject', 'review'))

lo_c, hi_c = cost_optimal_cutoffs(p_calib, y_cal_arr, loan_cal, REVIEW_COST, REVIEWER_MISS, REVIEWER_FALSE_REJECT)
opt_tiers = apply_cutoffs(p_te, lo_c, hi_c)
opt_label = f'Cost-optimal cutoffs [{lo_c:.3f}, {hi_c:.3f}]'

df_tiers = pd.DataFrame(tier_rows(static_tiers, y_te, yr_te, 'Static Conformal')
                        + tier_rows(mond_tiers, y_te, yr_te, 'Mondrian Conformal')
                        + tier_rows(opt_tiers, y_te, yr_te, opt_label))
print(f"Cost-optimal cutoffs fit on 2012 under the base cost model: approve if p <= {lo_c:.3f}, reject if p >= {hi_c:.3f}.\n")
print("Default rate by method / tier / year:")
print(df_tiers.pivot_table(index=['Method', 'Tier'], columns='Year', values='DefaultRate').round(4).to_string())
print("\nShare of loans by method / tier / year (methods review different shares -- read costs, not tiers, across methods):")
print(df_tiers.pivot_table(index=['Method', 'Tier'], columns='Year', values='Pct_of_year').round(4).to_string())

approve_by_year = df_tiers[(df_tiers.Method == 'Static Conformal') & (df_tiers.Tier == 'approve') & (df_tiers.Year != 'All')]
print("\nStatic-conformal auto-approve default rate by year (the trend pooling hides):")
for _, r in approve_by_year.iterrows():
    print(f"  {int(r.Year)}: {r.DefaultRate:.1%} of auto-approved loans actually default")

n_te = len(y_te)
policies = {'Static Conformal': static_tiers, 'Mondrian Conformal': mond_tiers, opt_label: opt_tiers,
            'Approve all': np.full(n_te, 'approve', dtype=object),
            'Reject all':  np.full(n_te, 'reject',  dtype=object),
            'Review all':  np.full(n_te, 'review',  dtype=object)}

cost_rows = []
for name, tiers in policies.items():
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(n_te, bool) if yr == 'All' else yr_te == yr
        c = strategy_cost(tiers[m], y_te[m], loan_te[m])
        cost_rows.append(dict(Method=name, Year=yr, N=int(m.sum()), TotalCost=c.sum(), CostPerLoan=c.mean()))
df_cost = pd.DataFrame(cost_rows)
print(f"\nIllustrative cost per loan (LGD={LOSS_GIVEN_DEFAULT:.0%}, foregone margin={FOREGONE_MARGIN:.0%}, review fee={REVIEW_COST:.0f}, "
      f"reviewer misses {REVIEWER_MISS:.0%} of defaulters / wrongly rejects {REVIEWER_FALSE_REJECT:.0%} of good loans -- NOT real economics):")
cost_pivot = df_cost.pivot_table(index='Method', columns='Year', values='CostPerLoan')
print(cost_pivot.round(2).to_string())
print("Cheapest policy by year:", cost_pivot.idxmin().to_dict())

# ---- (d) sensitivity: the ranking depends on review cost and reviewer accuracy (cost-optimal cutoffs re-fit per cell) ----
sens = []
for rc in (75.0, 250.0, 500.0):
    for miss in (0.0, 0.3, 0.6):
        lo_s, hi_s = cost_optimal_cutoffs(p_calib, y_cal_arr, loan_cal, rc, miss, REVIEWER_FALSE_REJECT)
        cell_policies = {**policies, 'Cost-optimal cutoffs (re-fit)': apply_cutoffs(p_te, lo_s, hi_s)}
        cell_policies.pop(opt_label)
        for name, tiers in cell_policies.items():
            sens.append(dict(review_cost=rc, reviewer_miss=miss, Policy=name,
                             CostPerLoan=strategy_cost(tiers, y_te, loan_te, rc, miss, REVIEWER_FALSE_REJECT).mean()))
df_sens = pd.DataFrame(sens)
sens_pivot = df_sens.pivot_table(index=['review_cost', 'reviewer_miss'], columns='Policy', values='CostPerLoan')
print("\nSensitivity of pooled 2013-2015 cost per loan to review cost and reviewer miss-rate:")
print(sens_pivot.round(0).to_string())
print("Cheapest policy per cell:")
print(sens_pivot.idxmin(axis=1).to_string())
print("\nTakeaway: conformal thresholds only matter if they beat (i) thresholds fit directly to the cost and (ii) the trivial policies;")
print("the ranking moves with reviewer quality and review cost -- read the grid, not a single row.")

df_tiers.round(4).to_csv(f'{OUTDIR}/tables/20_decision_tiers_by_year.csv', index=False)
df_cost.round(2).to_csv(f'{OUTDIR}/tables/20_decision_cost_by_year.csv', index=False)
df_sens.round(2).to_csv(f'{OUTDIR}/tables/20_cost_sensitivity.csv', index=False)

## 20b. Fairer conformal baseline and cost-model sanity check
Two checks on §20. (i) Static conformal was run only at α = 0.10, whereas the cost-optimal cutoffs were *tuned* to the cost — an unequal contest. Static conformal is a one-parameter family of symmetric cutoffs $[1-\hat q,\hat q]$, so here α is also **tuned on 2012** to minimise cost, then frozen. (ii) The base cost model has foregone margin 8 % against loss-given-default 60 %; approving everything then costs more than rejecting everything unless the margin exceeds LGD·ȳ/(1−ȳ), which depends on the default rate of each year. We print that break-even per year and rerun the policy comparison over a margin grid.

In [ ]:
# (i) conformal with alpha tuned on 2012 (in-sample for the cost, exactly like the cost-optimal cutoffs)
def sym_tiers(p, q):
    return np.where(p < 1 - q, 'approve', np.where(p > q, 'reject', 'review'))

a_rows = []
for a_ in np.round(np.r_[0.002, 0.005, np.arange(0.01, 0.41, 0.01)], 3):
    q_ = conf_q(cal_scores, a_)
    if not (0.5 < q_ < np.inf):
        continue
    a_rows.append(dict(alpha=a_, q=q_, cost_2012=strategy_cost(sym_tiers(p_calib, q_), y_cal_arr, loan_cal).mean()))
df_alpha = pd.DataFrame(a_rows)
a_star = float(df_alpha.loc[df_alpha.cost_2012.idxmin(), 'alpha']); q_star = conf_q(cal_scores, a_star)
print(f"alpha tuned on 2012 cost: alpha* = {a_star:.3f} (q = {q_star:.3f}); alpha = 0.10 was q = {q_static:.3f}")
if a_star in (df_alpha.alpha.min(), df_alpha.alpha.max()):
    print("WARNING: alpha* sits on the edge of the grid -- widen the grid before reading this row.")

tuned_tiers = sym_tiers(p_te, q_star)
cmp_rows = []
for name, tiers in (('Static conformal, alpha=0.10', static_tiers), (f'Static conformal, alpha*={a_star:.3f} (tuned on 2012)', tuned_tiers),
                    ('Mondrian conformal, alpha=0.10', mond_tiers), (opt_label, opt_tiers)):
    for yr in (2013, 2014, 2015, 'All'):
        m = np.ones(len(y_te), bool) if yr == 'All' else yr_te == yr
        cmp_rows.append(dict(Policy=name, Year=yr, CostPerLoan=strategy_cost(tiers[m], y_te[m], loan_te[m]).mean()))
print(pd.DataFrame(cmp_rows).pivot_table(index='Policy', columns='Year', values='CostPerLoan').round(1).to_string())

# (ii) break-even margin and margin sensitivity (cost-optimal cutoffs re-fit per margin)
print(f"\nBreak-even foregone margin LGD*ybar/(1-ybar) (approve-all == reject-all); the base model uses {FOREGONE_MARGIN}:")
for yr_ in (2012, 2013, 2014, 2015):
    yb = float(y_all[yr_all == yr_].mean())
    print(f"  {yr_}: default rate {yb:.3f} -> break-even margin {LOSS_GIVEN_DEFAULT * yb / (1 - yb):.3f}")
_orig_margin = FOREGONE_MARGIN
m_rows = []
for fm in (0.08, 0.25, 0.50):
    FOREGONE_MARGIN = fm                      # case_costs() reads this module-level constant at call time
    lo_m, hi_m = cost_optimal_cutoffs(p_calib, y_cal_arr, loan_cal, REVIEW_COST, REVIEWER_MISS, REVIEWER_FALSE_REJECT)
    pols = {'Static conformal a=0.10': static_tiers, 'Mondrian conformal a=0.10': mond_tiers,
            'Cost-optimal (re-fit)': apply_cutoffs(p_te, lo_m, hi_m),
            'Approve all': np.full(n_te, 'approve', dtype=object), 'Reject all': np.full(n_te, 'reject', dtype=object),
            'Review all': np.full(n_te, 'review', dtype=object)}
    for name, t_ in pols.items():
        m_rows.append(dict(foregone_margin=fm, Policy=name, CostPerLoan=strategy_cost(t_, y_te, loan_te).mean()))
FOREGONE_MARGIN = _orig_margin
mp = pd.DataFrame(m_rows).pivot_table(index='foregone_margin', columns='Policy', values='CostPerLoan')
print("\nPooled 2013-2015 cost per loan by foregone margin (reviewer miss 30 %, false-reject 10 %, fee 75):")
print(mp.round(0).to_string()); print("Cheapest:", mp.idxmin(axis=1).to_dict())
df_alpha.round(4).to_csv(f'{OUTDIR}/tables/20b_alpha_tuned_on_2012.csv', index=False)
pd.DataFrame(cmp_rows).round(2).to_csv(f'{OUTDIR}/tables/20b_policy_cost_alpha_tuned.csv', index=False)
mp.round(2).to_csv(f'{OUTDIR}/tables/20b_margin_sensitivity.csv')

## 20e. Monitoring: how early is the loss of exchangeability detectable?
The conclusions say the results support *monitoring* rather than a validated fix: the predicted-vs-realised default gap was visible years before coverage failed. This section tests that claim. Five monitors are run causally, month by month, after deployment:

| monitor | needs labels? | statistic |
|---|---|---|
| PSI > 0.10 / > 0.25 | no | population stability index of the model score, trailing 3 months vs. the 2012 calibration scores (credit-industry rules of thumb) |
| KS p < 1e-3 | no | two-sample Kolmogorov–Smirnov test of the same two score samples |
| Coverage test | yes (delay $d$) | one-sided $z$-test (0.1 % level) that static-set coverage over the last 6 *matured* issue months is $\ge 0.90$, with variance $\alpha(1-\alpha)(1/n_{\text{window}}+1/n_{\text{cal}})$ so that the noise of the calibration threshold is included |
| Calibration-in-the-large test | yes (delay $d$) | two-sided $z$-test (0.1 % level) that mean(observed − predicted) over the same cohort equals the *calibration set's own* mean(observed − predicted) |
| Conformal martingale ≥ 100 | yes (delay $d$) | simple-mixture test martingale on **online** smoothed conformal p-values (each score ranked against the calibration set and all earlier stream scores, which makes them i.i.d. uniform under exchangeability); Ville's inequality bounds the false-alarm probability by 1 %. Labels are processed in issue order, so the revealed set is always a prefix |

A monitor is useful only if its **false-alarm rate is small**, so each is also run on 50 permutations of the 2012–2015 rows of the real file (an exchangeable stream by construction). Labels of a loan issued in month $m'$ are usable from month $m'+d+1$.

In [ ]:
# ---------- §20e Monitoring: how early is the loss of exchangeability detectable, with and without labels? ----------
import bisect
from scipy.stats import ks_2samp, kstest
from scipy.special import logsumexp

EPS_GRID = np.linspace(0.05, 0.95, 19)
def online_conformal_pvalues(cal_s, s, rng):
    """Smoothed ONLINE conformal p-values: each stream score is ranked against the calibration scores AND all earlier stream scores. If the whole sequence is
    exchangeable these are i.i.d. U(0,1) (Vovk et al.), which is what the martingale needs. (Ranking against the fixed calibration set alone is NOT enough: given the
    calibration set the p-values are i.i.d. with a slightly non-uniform law, and over thousands of points a martingale picks that up and false-alarms.)"""
    arr = sorted(np.asarray(cal_s, float).tolist()); out = np.empty(len(s)); U = rng.random(len(s))
    for t, v in enumerate(np.asarray(s, float)):
        hi, lo = bisect.bisect_right(arr, v), bisect.bisect_left(arr, v)
        out[t] = ((len(arr) - hi) + U[t] * (hi - lo + 1)) / (len(arr) + 1); bisect.insort(arr, v)
    return out

_r = np.random.default_rng(0); _perm = _r.permutation(np.r_[p_calib, p_te]); _pv = online_conformal_pvalues(_perm[:len(p_calib)], _perm[len(p_calib):], _r)   # real scores, shuffled -> exchangeable
assert kstest(_pv, 'uniform').pvalue > 1e-3, "online conformal p-values are not uniform under exchangeability"

def log_mixture_martingale(pv):
    """Simple-mixture conformal test martingale: average of the power martingales prod eps * p_i^(eps-1) over a grid of eps. Ville: P(sup M >= c) <= 1/c."""
    lp = np.log(np.clip(pv, 1e-12, 1.0))
    return logsumexp(np.stack([np.cumsum(np.log(e) + (e - 1) * lp) for e in EPS_GRID]), axis=0) - np.log(len(EPS_GRID))

def psi_fn(edges, ref_frac, cur):
    c = np.clip(np.bincount(np.searchsorted(edges, cur), minlength=len(edges) + 1) / len(cur), 1e-4, None); r = np.clip(ref_frac, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))

def run_monitors(p_cal, y_cal, p, y, m, delays=(0, 6, 12), trail=3, win=6, n_min=200, z_cov=-3.09, z_citl=3.29, M_thr=100.0, seed=0):
    """Causal monitors on a monthly stream. Label-free: PSI and KS of the model score in the trailing `trail` months (full windows only) against the calibration scores.
    Label-based (labels of a loan issued in month m' are usable from month m'+delay+1): (i) one-sided z-test that coverage of the STATIC sets over the last `win` matured
    issue months is >= 0.90, whose variance includes the noise of the calibration-set threshold (0.09/n_cal) as well as of the window; (ii) two-sided z-test that the
    cohort's mean(y - p) equals the CALIBRATION set's own mean(y - p) (not 0: the model is not exactly calibrated even on 2012); (iii) conformal test martingale on online
    p-values. The z cut-offs correspond to 0.1 % tests, to leave room for the up-to-36 monthly looks. Returns first alarm month per monitor + histories."""
    rng = np.random.default_rng(seed); cs = true_label_scores(p_cal, y_cal); q = conf_q(cs, alpha); s = true_label_scores(p, y); hit = (s <= q)
    edges = np.quantile(p_cal, np.linspace(0, 1, 11)[1:-1]); ref = np.bincount(np.searchsorted(edges, p_cal), minlength=len(edges) + 1) / len(p_cal)
    logM = log_mixture_martingale(online_conformal_pvalues(cs, s, rng)); n_c = len(p_cal); res_c = y_cal - p_cal; months = np.unique(m); first, hist = {}, []
    def fire(name, mm): first.setdefault(name, mm)
    for mm in months:
        cur = p[(m > mm - trail) & (m <= mm)]; row = dict(month=mm, psi=np.nan, ks_p=np.nan)
        if mm - months[0] >= trail - 1 and len(cur) >= 150:
            row['psi'] = psi_fn(edges, ref, cur); row['ks_p'] = ks_2samp(p_cal, cur).pvalue
            if row['psi'] > 0.10: fire(('PSI > 0.10', '-'), mm)
            if row['psi'] > 0.25: fire(('PSI > 0.25', '-'), mm)
            if row['ks_p'] < 1e-3: fire(('KS p < 1e-3', '-'), mm)
        for d in delays:
            sel = (m > mm - d - 1 - win) & (m <= mm - d - 1); row[f'cov_d{d}'] = np.nan
            if sel.sum() >= n_min:
                row[f'cov_d{d}'] = hit[sel].mean()
                nw = int(sel.sum()); zc = (hit[sel].mean() - (1 - alpha)) / np.sqrt(alpha * (1 - alpha) * (1 / nw + 1 / (n_c + 2)))
                if zc < z_cov: fire(('Coverage test', d), mm)
                rw = y[sel] - p[sel]; zr = (rw.mean() - res_c.mean()) / np.sqrt(rw.var(ddof=1) / nw + res_c.var(ddof=1) / n_c)
                if abs(zr) > z_citl: fire(('Calibration-in-the-large test', d), mm)
        hist.append(row)
    idx = np.where(logM >= np.log(M_thr))[0]
    for d in delays:
        if len(idx): first[('Conformal martingale >= 100', d)] = m[idx[0]] + d + 1
    return first, pd.DataFrame(hist), logM

def t_to_date(t): return f"{2007 + (int(t) - 1) // 12}-{(int(t) - 1) % 12 + 1:02d}"
MON_ORDER = [('PSI > 0.10', '-'), ('PSI > 0.25', '-'), ('KS p < 1e-3', '-')] + [(n, d) for n in ('Coverage test', 'Calibration-in-the-large test', 'Conformal martingale >= 100') for d in (0, 6, 12)]

# ---- real file: alarms, plus a permutation null (rows of 2012-2015 shuffled, so the stream IS exchangeable with the calibration set) ----
first_re, hist_re, logM_re = run_monitors(p_calib, y_calib.values, p_te, y_te, m_te)
pool = np.where(yr_all >= 2012)[0]; assert len(pool) == len(p_calib) + len(p_te)
null_hits = {k: 0 for k in MON_ORDER}; N_PERM = 50
for r in range(N_PERM):
    pm = np.random.default_rng(500 + r).permutation(pool); ci, si = pm[:len(p_calib)], pm[len(p_calib):]
    f_, _, _ = run_monitors(p_all[ci], y_all[ci], p_all[si], y_all[si], m_te, seed=r)
    for k in f_: null_hits[k] += 1
rows = [dict(monitor=k[0], delay_months=k[1], first_alarm=(t_to_date(first_re[k]) if k in first_re else 'none'),
             months_after_deployment=(int(first_re[k] - m_te.min() + 1) if k in first_re else np.nan), null_false_alarm_rate=null_hits[k] / N_PERM) for k in MON_ORDER]
df_mon = pd.DataFrame(rows)
print(f"Monitors on the real stream (deployment starts {t_to_date(m_te.min())}); null = {N_PERM} permutations of 2012-2015 rows (any alarm over the 36 months):")
print(df_mon.to_string(index=False))
lf = [first_re[k] for k in MON_ORDER[:3] if k in first_re]; lb12 = [first_re[k] for k in MON_ORDER if k[1] == 12 and k in first_re]
if lf and lb12: print(f"\nEarliest label-free alarm {t_to_date(min(lf))} vs earliest alarm requiring 12-month-old labels {t_to_date(min(lb12))}: lead of {int(min(lb12) - min(lf))} months.")
print("Reading: a monitor is only useful if its null false-alarm rate is small. The Coverage / CITL tests are 0.1 % tests applied up to 36 times with overlapping windows, so their false-alarm rate is not exactly 3.6 % and is estimated here; the martingale controls false alarms by Ville's inequality (<= 1 %).")
print("`hist_re` holds the monthly trailing coverage and PSI; compare the alarm dates with the dates at which coverage actually leaves 0.90.")

fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True); xm = hist_re.month.values
l1, = ax[0].plot(xm, hist_re.psi, color='tab:purple', label='PSI (left axis)'); ax[0].axhline(0.10, ls='--', c='tab:purple', alpha=.5); ax[0].axhline(0.25, ls=':', c='tab:purple', alpha=.5); ax[0].set_ylabel('PSI')
a2 = ax[0].twinx(); l2, = a2.plot(xm, -np.log10(hist_re.ks_p), color='tab:green', label='-log10 KS p (right axis)'); a2.axhline(3, ls='--', c='tab:green', alpha=.5); a2.set_ylabel('-log10 KS p'); a2.grid(False)
ax[0].legend(handles=[l1, l2], fontsize=8, loc='upper left'); ax[0].set_title('Label-free monitors: model-score drift against 2012 (dashed = alarm levels)', fontweight='bold')
for d_, c_ in ((0, 'tab:blue'), (12, 'tab:red')):
    ax[1].plot(xm, hist_re[f'cov_d{d_}'], color=c_, label=f'static coverage, last 6 matured months (label delay {d_})')
    if ('Coverage test', d_) in first_re: ax[1].axvline(first_re[('Coverage test', d_)], color=c_, ls='--', lw=1, label=f'coverage-test alarm (delay {d_})')
ax[1].axhline(0.9, ls='--', c='k'); ax[1].set_ylabel('coverage'); ax[1].legend(fontsize=7, loc='lower left'); ax[1].set_title('Label-based monitor: coverage of the static sets', fontweight='bold')
mths = np.unique(m_te); ax[2].plot(mths, [logM_re[m_te == mm][-1] / np.log(10) for mm in mths], color='tab:orange', marker='.'); ax[2].axhline(2, ls='--', c='k')
for d_, c_ in ((0, 'tab:blue'), (12, 'tab:red')):
    if ('Conformal martingale >= 100', d_) in first_re: ax[2].axvline(first_re[('Conformal martingale >= 100', d_)], color=c_, ls='--', lw=1, label=f'alarm with label delay {d_}')
ax[2].legend(fontsize=8, loc='upper left'); ax[2].set_ylabel('log10 martingale (month end)'); ax[2].set_xlabel('month index (issue date)'); ax[2].set_title('Conformal test martingale (alarm at 100)', fontweight='bold')
plt.tight_layout(); save_fig('20e_monitoring_real'); plt.show()
df_mon.to_csv(f'{OUTDIR}/tables/20e_monitoring_real.csv', index=False); hist_re.round(5).to_csv(f'{OUTDIR}/tables/20e_monitoring_monthly.csv', index=False)


## 20d. Paper artefacts
Writes booktabs LaTeX tables and CSVs to `tables/latex/` and prints the **key numbers** used in the abstract and results, computed from the live objects so that the text cannot drift from the tables again. Also prints a draft results sentence for the abstract. Requires §5, §12b, §13b, §15, §15c, §16c, §19c and §20e to have been run.

In [ ]:
# ---------- §20d Paper artefacts: LaTeX tables and a 'key numbers' block generated from the objects above (no hand-typed figures) ----------
os.makedirs(f'{OUTDIR}/tables/latex', exist_ok=True)
def to_latex_simple(d, caption, label, fmt='{:.3f}'):
    def cell(v):
        if isinstance(v, (float, np.floating)): return '--' if not np.isfinite(v) else fmt.format(v)
        return str(v).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')
    head = ' & '.join(str(c).replace('_', r'\_') for c in d.columns)
    body = '\n'.join(' & '.join(cell(v) for v in r) + r' \\' for r in d.itertuples(index=False))
    return ('\\begin{table}[t]\n\\centering\n\\caption{' + caption + '}\n\\label{' + label + '}\n\\begin{tabular}{' + 'l' * len(d.columns) + '}\n\\toprule\n'
            + head + ' \\\\n\\midrule\n' + body + '\n\\bottomrule\n\\end{tabular}\n\\end{table}\n')

need = ['df_perf', 'df_cov', 'diag', 'df_grid', 'df_ls', 'df_blk', 'df_wf', 'df_rv', 'df_slice_summary', 'df_nested', 'df_mon']
missing = [n for n in need if n not in globals()]
assert not missing, f"run the sections that create {missing} first"

t1 = (df_perf.rename(columns={'Year': 'year'}).merge(df_cov[['Year', 'N', 'Empirical Coverage', 'CI Low (95%)', 'CI High (95%)', 'Avg Set Size']]
      .rename(columns={'Year': 'year'}), on='year').merge(diag[['Year', 'mean_pred_risk', 'actual_default']].rename(columns={'Year': 'year'}), on='year'))
t2 = df_grid[['method', 'delay', 'cov_2013', 'cov_2014', 'cov_2015', 'size_all', 'share_within_3pp']]
t4 = df_ls[df_ls.year != 'All'].pivot_table(index='method', columns='year', values='cov').reset_index()
for name, d_, cap in (('t1_point_and_conformal_by_year', t1, 'Discrimination, calibration and static split-conformal coverage by year (calibration year 2012 is in-sample for coverage).'),
                      ('t2_adaptive_methods', t2, 'Adaptive and recalibrating methods under label delay (months); frozen base model, $\\alpha=0.10$.'),
                      ('t4_label_shift_conformal', t4, 'Label-shift-weighted conformal coverage by test year.')):
    open(f'{OUTDIR}/tables/latex/{name}.tex', 'w').write(to_latex_simple(d_, cap, 'tab:' + name)); d_.to_csv(f'{OUTDIR}/tables/latex/{name}.csv', index=False)
print(f"Wrote {len(os.listdir(f'{OUTDIR}/tables/latex'))} files to {OUTDIR}/tables/latex/")

_gD = G_DELAY[1]; _ACI = f'ACI, gamma={_gD}'; _TR = f'Quantile tracker, eta={_gD}'
# ---- key numbers for the abstract / results text, computed from the objects above ----
cv = df_cov.set_index('Year')['Empirical Coverage']; lo_ = df_cov.set_index('Year')['CI Low (95%)']; hi_ = df_cov.set_index('Year')['CI High (95%)']
g_ = lambda meth, d_, col: float(df_grid[(df_grid.method == meth) & (df_grid.delay == d_)][col].iloc[0])
print("\nKEY NUMBERS (paste from here, do not retype):")
print(f" - Static split conformal coverage 2013/2014/2015: {cv[2013]:.1%} / {cv[2014]:.1%} / {cv[2015]:.1%}  (2015 Wilson 95% CI {lo_[2015]:.1%}-{hi_[2015]:.1%})")
b18 = df_blk[(df_blk.year == 2015) & (df_blk.resampling == 'month-block bootstrap')].iloc[0]
print(f" - 2015 gap, month-block bootstrap 95% interval: [{b18.gap_lo_pp:+.2f}, {b18.gap_hi_pp:+.2f}] pp")
print(f" - Calibration-in-the-large gap (actual - predicted default rate) 2013/2014/2015: " + " / ".join(f"{100 * (diag.set_index('Year').loc[y_, 'actual_default'] - diag.set_index('Year').loc[y_, 'mean_pred_risk']):+.1f} pp" for y_ in (2013, 2014, 2015)))
print(f" - Retraining variance of 2015 static coverage over {len(df_rv)} seeds: {df_rv.cov_2015.mean():.3f} +- {df_rv.cov_2015.std():.3f}")
print(f" - 2015 coverage, delay 12: static {g_('Static (2012 threshold)', '-', 'cov_2015'):.3f} | ACI g={_gD:g} {g_(_ACI, 12, 'cov_2015'):.3f} | rolling recal W=12 {g_('Rolling recalibration, W=12 mo', 12, 'cov_2015'):.3f} | tracker eta={_gD:g} {g_(_TR, 12, 'cov_2015'):.3f}")
ls18 = df_ls[df_ls.year == 2015].set_index('method')['cov']
print(" - 2015 coverage, label-free prior weighting: " + " | ".join(f"{k.replace('Label-shift weighted, ', 'LS ').split(' (')[0]} {v:.3f}" for k, v in ls18.items()))
w_ = df_slice_summary[df_slice_summary.family == 'risk decile (p_hat)'].set_index('method')['worst_cov']
print(" - Worst risk-decile coverage: " + " | ".join(f"{k} {v:.3f}" for k, v in w_.items()))

# ---- auto-generated results sentence for the abstract (numbers only; edit wording, not figures) ----
n18 = df_nested[df_nested.test_year == 2015].set_index('method')
sent = (f"In 2015, static split conformal covers {cv[2015]:.1%} against a nominal 90 %; with labels delayed 12 months the frozen-model methods reach "
        f"{g_('Rolling recalibration, W=12 mo', 12, 'cov_2015'):.1%} (sliding-window recalibration), {g_(_ACI, 12, 'cov_2015'):.1%} (ACI, gamma {_gD:g}) and "
        f"{g_(_TR, 12, 'cov_2015'):.1%} (quantile tracking, eta {_gD:g}); with step sizes selected walk-forward instead of in hindsight, ACI (immediate feedback) reaches "
        f"{n18.loc['ACI gamma', 'cov_selected']:.1%} and DtACI {n18.loc['DtACI eta', 'cov_selected']:.1%}. Label-free prior-shift weighting gives "
        f"{ls18.iloc[1]:.1%} (EM) and {ls18.iloc[2]:.1%} (BBSE) against {ls18.iloc[3]:.1%} with the oracle prior.")
print("\nDRAFT RESULTS SENTENCE (abstract item 5; check every claim against the tables before using):\n" + sent)
mon = df_mon.set_index(['monitor', 'delay_months'])
def _m(k): r = mon.loc[k]; return f"{r.first_alarm} ({r.months_after_deployment:.0f} mo after deployment; null false-alarm rate {r.null_false_alarm_rate:.0%})" if r.first_alarm != 'none' else f"no alarm (null false-alarm rate {r.null_false_alarm_rate:.0%})"
print("\nMONITORING (real stream): PSI > 0.10: " + _m(('PSI > 0.10', '-')) + " | coverage test, delay 12: " + _m(('Coverage test', 12)) + " | martingale, delay 12: " + _m(('Conformal martingale >= 100', 12)))


## 21. Conclusions

> Write this section **after** running the notebook, from the tables in `outputs/tables` and the §20d key-numbers printout. No figure below was produced on your data; each bullet is a question to answer, not a finding.

1. **Discrimination vs calibration.** How do AUC and Brier/log-loss/calibration-in-the-large move from 2013 to 2015? (§3, §15)
2. **Does static split-conformal coverage hold out-of-time?** Per-year coverage with Wilson and block-bootstrap intervals; compare with the i.i.d. control (§5, §14, §16, §16b, §16c).
3. **What kind of shift is it?** Covariate vs prior vs concept drift (§15, §15b, §15c). Do not use the train-pool domain AUC as evidence: it is confounded by `int_rate` (§15b).
4. **ACI.** Its long-run coverage is a by-construction property for the unclipped algorithm with immediate feedback (§11b); report set size, $\alpha_t$ range and $\gamma$ sensitivity alongside coverage.
5. **Delayed labels.** Which methods keep coverage at 12/24/36-month delay (§12, §12b, §16b)? Remember γ, η and ρ are tuned on the test years unless you use the walk-forward versions (§16c, §19c).
6. **Who is under-covered?** Class-conditional and slice coverage (§13, §13b).
7. **Robustness.** Base learner, `int_rate` ablation, α sweep (§17, §18, §18c–e).
8. **Decision utility.** Conformal tiers vs cost-optimal cutoffs under the *illustrative* cost model (§20, §20b); the cost parameters are assumptions, not estimates.
9. **Monitoring.** Detection delay and false-alarm rate (§20e).

**Limitations to state in the paper:** single lender and a single 36-month population; labels for loans issued in 2016+ are not used (maturity); calibration on one year; the cost model is illustrative; hyper-parameters of ACI/DtACI/recency weighting are tuned on the evaluation years unless walk-forward results are quoted.


## 21b. Evidence map (claim → section → file in `outputs/tables`)
Use this table to check every sentence of the paper against its source. Cells with an empty output must be run first.

| Claim (Conclusion #) | Section | Files |
|---|---|---|
| Calibration decays more than discrimination (1) | 3, 15 | `03_yearly_point_metrics`, `15_shift_diagnosis` |
| Static coverage by test year (3) | 5, 14, 16, 16b, 16c | `05_yearly_coverage_with_CI`, `16b_combined_bootstrap_gap`, `16c_block_bootstrap`, `16c_retraining_variance` |
| ACI's ≈ 90 % is by construction; clipping and delay break it (4, 5) | 11, 11b, 12, 12b | `11b_aci_clipped_vs_unclipped`, `11b_aci_delay_clipped_vs_unclipped`, `12b_recalibration_tracker_grid` |
| Marginal coverage hides defaults; Mondrian moves the failure (6) | 13, 13b | `13_mondrian_vs_static`, `13b_slice_coverage`, `13b_worst_slice_summary` |
| Dependence on α (7) | 17 | — (printed) |
| Kind of shift: label drift vs covariate vs prior (2) | 15, 15b, 15c | `15b_domain_diagnostics_vs_calibration`, `15c_label_shift_conformal` |
| Decision framework and cost model (8) | 20, 20b | `20_decision_tiers_by_year`, `20_decision_cost_by_year`, `20_cost_sensitivity`, `20b_*` |
| Robust to base learner and to `int_rate` (9) | 18a, 18c, 18d, 18e | `18a_logreg_vs_lightgbm`, `18d_conformal_three_base_models`, `18e_int_rate_ablation` |
| Shift-aware alternatives and honest tuning (10, 13) | 18b, 19, 19c, 16c | `18b_posthoc_calibration`, `19a_dtaci`, `19b_barber_weighted`, `19c_nested_selection`, `16c_walk_forward_gamma` |
| Retrain-and-recalibrate under label lag (11) | 16b | `16b_retrain_recalibrate` |
| Monitoring (14) | 20e | `20e_monitoring_real`, `20e_monitoring_monthly` |
| Every figure and table in the paper | 20d | `tables/latex/*.tex`, `figures/*.pdf` |

## 22. References and pre-submission checklist

**Core references cited in this notebook** (verify page/volume details before submission)
- Vovk, Gammerman & Shafer (2005). *Algorithmic Learning in a Random World.* Springer.
- Lei, G'Sell, Rinaldo, Tibshirani & Wasserman (2018). Distribution-free predictive inference for regression. *JASA* 113(523).
- Sadinle, Lei & Wasserman (2019). Least ambiguous set-valued classifiers with bounded error levels. *JASA* 114(525). *(the score $1-\hat p(y\mid x)$ used here)*
- Vovk (2012). Conditional validity of inductive conformal predictors. *ACML*. *(Mondrian / class-conditional conformal)*
- Tibshirani, Barber, Candès & Ramdas (2019). Conformal prediction under covariate shift. *NeurIPS*.
- Gibbs & Candès (2021). Adaptive conformal inference under distribution shift. *NeurIPS*. *(ACI)*
- Gibbs & Candès (2024). Conformal inference for online prediction with arbitrary distribution shifts. *JMLR* 25 (arXiv 2208.08401). *(DtACI)*
- Barber, Candès, Ramdas & Tibshirani (2023). Conformal prediction beyond exchangeability. *Ann. Statist.* 51(2).
- Angelopoulos, Candès & Tibshirani (2023). Conformal PID control for time series prediction. *NeurIPS*. *(not implemented)*

**Before submitting**
1. **Run all cells** on the real file; delete stale CSVs in `outputs/tables` from earlier runs; re-read §21 against §20d's key numbers.
2. **Related work.** No credit-scoring-specific conformal / drift literature is cited here; search Scopus / Google Scholar / arXiv for conformal × credit scoring × (distribution shift | concept drift | label delay) and cite what you verify yourself.
3. **Data.** Cite the LendingClub source and the Kaggle mirror (N. George, "All Lending Club loan data"); archive the exact processed CSV (e.g. Zenodo) and pin the SHA-256 printed by the loading cell. Report `MAX_PER_MONTH`, the seed, and the selection rules of `prepare_lending_club.py`.
4. **Selection effects.** Only 36-month loans issued through 2015 with a final status are used; say so, and note that `int_rate` is the lender's own risk price (§2b, §18e).
5. **Hyper-parameters.** η, ρ and the DtACI γ grid are tuned on evaluation years in §19; §19c gives walk-forward selection for η, ρ. Quote the walk-forward numbers.
6. **Optional full-data run.** Set `MAX_PER_MONTH=0` (slow: the online monitors and weighted conformal scale with the number of test loans).
